# NCTB-BERT — A Grade-Referenced Curriculum Encoder for Bangladeshi Educational Text

**Single-notebook, Kaggle-ready research pipeline** for the paper
*"NCTB-BERT: Grade-Referenced Adaptive-Difficulty Curriculum Pretraining for Low-Resource Educational Language Understanding"*.

---

## 1. What is actually novel here (the methodology contribution)

Most domain-adaptive pretraining (DAPT) work shuffles the domain corpus uniformly. Curriculum-learning
work for LMs must *invent* a difficulty signal (loss, length, perplexity, gradient influence) because
no ground-truth ordering of text by conceptual difficulty exists.

**NCTB textbooks are different: they come with a human-expert, nationally standardised difficulty
ordering — the grade level.** A Class-3 science chapter and a Class-11 physics chapter were placed at
those grades by curriculum experts on the basis of conceptual load. That is a *free, externally
validated difficulty label* over 46,560 passages. No other large corpus has this.

The pipeline therefore contributes four things:

| # | Contribution | Why it is novel |
|---|---|---|
| **C1** | **GRADE-CL** — Grade-Referenced Adaptive Difficulty Curriculum Learning. A composite difficulty index anchored on the *human curriculum grade* and refined by intrinsic lexical/syntactic/conceptual features, driving a competence-based pacing function over pretraining. | First use of a *national curriculum's own grade ordering* as the ground-truth competence signal for LM pretraining. Prior CL work uses proxy signals; we use — and empirically validate against — the expert signal. |
| **C2** | **Grade-anchored validation of intrinsic difficulty proxies.** Because the grade label exists, we can measure how well length/TTR/rarity proxies (used by all prior CL papers) actually correlate with expert-assigned difficulty. | Turns the corpus into a *testbed* that retroactively evaluates the entire proxy-based CL literature. This alone is a publishable finding. |
| **C3** | **Grapheme-cluster-constrained Bengali vocabulary + FOCUS transfer**, evaluated as its own arm against the selected base encoder's native vocabulary. We report the measured *cluster-split rate* (fraction of tokens that strand a matra from its base) alongside fertility, rather than assuming cluster safety. | Cluster-split rate is a tokenizer-quality axis the Bengali literature reports informally at best; we make it a measured metric and tie it to a parameter-efficiency result. |
| **C4** | **NCTBench-Eval** — a leakage-safe, *book-level-split* benchmark of 4 tasks (grade-band, subject, chapter-boundary, passage retrieval) built from dataset metadata at zero annotation cost. | Chunk-level splits leak: adjacent chunks of the same chapter are near-duplicates. We split by **book**, and quantify how much prior-style chunk-level splitting inflates scores. |

**Base encoder is chosen empirically**, not assumed: Cell 16b runs an identical probe DAPT for each
MLM-head candidate (XLM-R, MuRIL, bangla-bert-base, bengali-bert) and picks the winner on held-out
loss. BanglaBERT/BanglishBERT are excluded because they are ELECTRA discriminators with no MLM head —
stated as a limitation, with an RTD-objective variant as scoped future work.

**Ablation arms shipped** (all runnable from one config flag): `grade_cl` (ours), `random` (standard DAPT),
`anti` (hard→easy), `length_cl` (classic proxy curriculum), `vocab_swap` (GRADE-CL on the NCTB
vocabulary, isolating C3 from C1), `no_dapt` (frozen base).
Optional extra objectives (OCR-confidence-weighted MLM, dual-OCR-view contrastive) are **implemented and
wired but disabled by default** — flip one flag each to add ablation arms.

---

## 2. How to run on Kaggle

0. **Run Cell 0 (the environment doctor) first, on its own.** Kaggle images sometimes ship a
   `transformers` build with a broken lazy-import table, which fails with
   `ValueError: Backend should be defined in the BACKENDS_MAPPING. Offending backend: keras_nlp`.
   Cell 0 detects this in a subprocess, and *only if it is actually broken* installs a known-good pin
   and restarts the kernel. If your environment is healthy it installs nothing — which matters,
   because unconditionally `pip install`-ing `transformers` on Kaggle is what causes the breakage in
   the first place. After a restart, just run the notebook again from the top.
   The pipeline needs **no third-party packages beyond what Kaggle already ships** — MinHash
   deduplication and the FOCUS auxiliary embedding space are implemented directly on
   numpy/scipy/scikit-learn rather than pulling in `datasketch` and `gensim`.
1. **Accelerator: GPU T4 ×2 — not P100.** Kaggle's current PyTorch is built without `sm_60`
   kernels, so on a P100 every CUDA launch dies with *"no kernel image is available for execution
   on the device"* at the first forward pass. Cell 1 now preflights this and tells you immediately.
   **Internet:** ON for the first run (downloads XLM-R + baselines);
   afterwards you can turn it off and point `CFG.model_cache` at a Kaggle Dataset of the cached models.
2. The notebook is **stage-checkpointed**. Every stage writes a marker to `/kaggle/working/nctb_bert/.stages/`.
   If a 12-hour session times out, just **re-run the whole notebook** — completed stages are skipped and it
   resumes. Save the `nctb_bert/` output folder as a Kaggle Dataset and mount it as `CFG.resume_from` to
   carry state across sessions.
3. **Budget guard:** `CFG.session_budget_hours` stops training gracefully before the Kaggle wall-clock limit
   and saves a resumable checkpoint.
4. First smoke-test with `CFG.smoke_test = True` (≈8 minutes end-to-end) before the full run.

**Suggested multi-session schedule** (30 GPU-h/week):

| Session | Stages | ~GPU-h |
|---|---|---|
| 1 | data → splits → difficulty → tokenizer → FOCUS init | 1.5 |
| 2 | pretrain arm `grade_cl` | 9 |
| 3 | pretrain arms `random`, `anti` | 9 |
| 4 | pretrain arm `length_cl` + intrinsic eval | 6 |
| 5 | downstream ×5 seeds ×6 models + stats + figures | 4 |


In [ ]:
# ============================================================================
# CELL 0 — ENVIRONMENT DOCTOR.  Run this FIRST, on its own.
# ----------------------------------------------------------------------------
# Kaggle images periodically ship a `transformers` build whose lazy-import table
# is internally inconsistent, which raises at import time:
#     ValueError: Backend should be defined in the BACKENDS_MAPPING.
#                 Offending backend: keras_nlp   (or tensorflow_text)
# It is a transformers-internal bug, not something wrong with your code, and the
# only fix is to move to a version that does not have it.
#
# This cell probes the CURRENT interpreter in a subprocess, and only if the probe
# fails does it install a known-good pin and restart the kernel. If nothing is
# broken it installs nothing at all -- which is the single most important thing,
# because blindly `pip install`-ing transformers on Kaggle is what breaks it.
# ============================================================================
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("WANDB_DISABLED", "true")
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")

_PROBE = r"""
import sys
try:
    import transformers, tokenizers
    from transformers import (AutoConfig, AutoTokenizer, AutoModel, AutoModelForMaskedLM,
                              AutoModelForSequenceClassification, PreTrainedTokenizerFast,
                              get_linear_schedule_with_warmup, set_seed)
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers, processors
    print("OK|%s|%s" % (transformers.__version__, tokenizers.__version__))
except Exception as e:
    print("FAIL|%s|%s" % (type(e).__name__, str(e)[:160])); sys.exit(1)
"""

def _probe():
    r = subprocess.run([sys.executable, "-c", _PROBE], capture_output=True, text=True)
    line = ([l for l in (r.stdout + r.stderr).splitlines() if l.startswith(("OK|", "FAIL|"))] or ["FAIL|?|?"])[-1]
    return r.returncode == 0, line

def _pip(*pkgs):
    print("installing:", " ".join(pkgs))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input", *pkgs], check=False)

ok, msg = _probe()
print("probe:", msg)

if not ok:
    # Known-good pins, newest first. 5.8.0 predates the BACKENDS_MAPPING regression;
    # 4.46.3 + tokenizers 0.20.3 is the long-term-stable fallback.
    # --no-deps is deliberate: without it pip may pull a different `torch` wheel,
    # and a torch built for other GPU architectures is what produces
    # "CUDA error: no kernel image is available for execution on the device".
    # Never let a transformers pin drag torch along.
    for pins in (["transformers==5.8.0"],
                 ["transformers==4.46.3", "tokenizers==0.20.3"]):
        _pip("--no-deps", *pins)
        ok, msg = _probe()
        print("probe after", pins, "->", msg)
        if ok: break
    if ok:
        print("\n" + "=" * 74)
        print("Environment repaired. RESTARTING THE KERNEL NOW.")
        print("When it comes back, run the notebook again from the top —")
        print("this cell will find a healthy environment and install nothing.")
        print("=" * 74, flush=True)
        import time as _t; _t.sleep(2)
        os.kill(os.getpid(), 9)          # Kaggle restarts the kernel automatically
    else:
        raise RuntimeError(f"could not repair the transformers install: {msg}")
else:
    print("environment is healthy — no installs performed")

# `regex` is the only genuinely optional dependency left (Unicode \\X grapheme
# clusters, which Bengali conjunct handling needs). Everything else in this
# notebook uses numpy / scipy / scikit-learn / torch / transformers only.
try:
    import regex  # noqa
except Exception:
    _pip("regex")


In [ ]:
# ============================================================================
# CELL 1 — Imports
# ============================================================================
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
import json, math, time, random, hashlib, shutil, unicodedata, itertools, gc, zlib
from pathlib import Path
from dataclasses import dataclass, field, asdict
from collections import Counter, defaultdict
from typing import List, Dict, Optional, Tuple, Any

import numpy as np
import pandas as pd
import regex as re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader, Sampler
from tqdm.auto import tqdm

import transformers
from transformers import (
    AutoConfig, AutoTokenizer, AutoModel, AutoModelForMaskedLM,
    AutoModelForSequenceClassification, PreTrainedTokenizerFast,
    get_linear_schedule_with_warmup, set_seed as hf_set_seed,
)
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers, processors

from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sps
from scipy import stats as sp_stats

def make_scaler(enabled: bool):
    """torch.cuda.amp.GradScaler is deprecated in torch>=2.4; prefer the new API."""
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled)

transformers.logging.set_verbosity_error()   # silence the per-load "LOAD REPORT" tables

print(f"python      : {sys.version.split()[0]}")
print(f"torch       : {torch.__version__}   cuda={torch.cuda.is_available()}")
print(f"transformers: {transformers.__version__}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU = torch.cuda.device_count()

def gpu_preflight():
    """
    Fail fast on a torch-build / GPU-architecture mismatch.

    Kaggle's PyTorch is currently built WITHOUT sm_60 kernels, so selecting the
    **P100** accelerator makes every CUDA kernel launch fail with
        CUDA error: no kernel image is available for execution on the device
    ...but only once you reach the first forward pass, an hour into the pipeline.
    This check catches it in two seconds instead.
    """
    if not torch.cuda.is_available():
        print("\n!! No GPU visible. Set Settings -> Accelerator -> 'GPU T4 x2'.")
        print("   The pipeline will run on CPU but pretraining will be impractically slow.\n")
        return
    arches = torch.cuda.get_arch_list()
    for i in range(N_GPU):
        p = torch.cuda.get_device_properties(i)
        sm = f"sm_{p.major}{p.minor}"
        print(f"  GPU{i}: {p.name}  {p.total_memory/1e9:.1f} GB  compute={sm}")
    p0 = torch.cuda.get_device_properties(0)
    sm0 = f"sm_{p0.major}{p0.minor}"
    try:                                  # actually launch a kernel
        (torch.randn(64, 64, device="cuda") @ torch.randn(64, 64, device="cuda")).sum().item()
        torch.cuda.synchronize()
        launched, err = True, None
    except Exception as e:
        launched, err = False, e
    if launched and sm0 in arches:
        print(f"  torch supports: {' '.join(arches)}  -> OK")
        return
    raise RuntimeError(
        "\n" + "=" * 78 +
        f"\nGPU / PyTorch ARCHITECTURE MISMATCH — this is an environment problem, not a bug.\n\n"
        f"  device            : {p0.name}  ({sm0})\n"
        f"  torch {torch.__version__} was built for: {' '.join(arches)}\n"
        f"  test kernel launch: {'ok' if launched else 'FAILED — ' + str(err)[:90]}\n\n"
        "FIX (10 seconds): Notebook -> Settings -> Accelerator -> **GPU T4 x2**, then\n"
        "re-run from the top. T4 is sm_75, is supported by every current torch build,\n"
        "has 2x16 GB, and is what this notebook's batch sizes are tuned for.\n\n"
        "The P100 is sm_60 and Kaggle's PyTorch no longer ships sm_60 kernels\n"
        "(Kaggle/docker-python issue #1546). Reinstalling an older CUDA 12.6 torch\n"
        "would also work but costs a ~2.5 GB download every session — not worth it.\n"
        + "=" * 78)

gpu_preflight()

# bf16 where the hardware has it (Ampere+); T4/sm_75 is fp16-only.
AMP_DTYPE = (torch.bfloat16
             if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
             else torch.float16)
print(f"  autocast dtype: {AMP_DTYPE}")


In [ ]:
# ============================================================================
# CELL 2 — CONFIG.  Every knob for every experiment lives here.
# ============================================================================
@dataclass
class CFG:
    # ---------- paths ----------
    data_root: str = "/kaggle/input/datasets/codderboy/nctb-final-dataset-class1-to-12/NCTB-dataset-clean"
    out_dir: str   = "/kaggle/working/nctb_bert"
    resume_from: Optional[str] = None      # e.g. "/kaggle/input/nctb-bert-run-v1/nctb_bert"
    model_cache: Optional[str] = None      # offline HF cache mounted as a Kaggle Dataset

    # ---------- run control ----------
    # Bump pipeline_version whenever data construction changes: stage markers are
    # namespaced by it, so every cached stage is invalidated and rebuilt. v2 fixed
    # the book_id collision and the empty-ds_train split arithmetic.
    pipeline_version: int = 4
    seed: int = 42
    smoke_test: bool = True                # <<< set False for the real run
    session_budget_hours: float = 10.5     # graceful stop before Kaggle's 12h wall
    force_stages: Tuple[str, ...] = ()     # e.g. ("tokenizer",) to redo one stage

    # ---------- Bangla-aware preprocessing ----------
    unicode_form: str = "NFKC"             # NFKC folds visually identical variants
    normalize_nukta: bool = True           # unify precomposed / base+nukta forms
    normalize_bn_digits: bool = False      # map Bengali digits to ASCII (off: preserves fidelity)
    strip_repeated_furniture: bool = True  # page headers/footers repeated across a book
    furniture_min_book_frac: float = 0.25  # an n-gram in >=25% of a book's chunks is furniture
    lex_coverage_min: float = 0.55         # drop chunks whose tokens are mostly corpus-unique junk
    lex_min_freq: int = 3
    max_recon_deviation: float = 0.15      # drop books whose recon_ratio is off by more than this
    pack_sequences: bool = True            # concatenate consecutive chunks within a chapter

    # ---------- corpus filtering ----------
    min_chars: int = 120
    max_chars: int = 6000
    min_ocr_agreement: float = 0.0         # 0.0 = keep all; agreement is used as a *weight*, not a filter
    dedup_threshold: float = 0.85          # MinHash Jaccard for near-duplicate removal
    drop_frontmatter_chapters: bool = True # chapter_no==1 chunk_index==1 is usually cover/imprint

    # ---------- splits (BOOK-level, leakage-safe) ----------
    # downstream_book_frac is a fraction of ALL books; ds_val_frac / ds_test_frac
    # are fractions of the DOWNSTREAM POOL, not of the whole corpus. The remainder
    # of the pool becomes ds_train.  (0.25 + 0.30 -> 0.45 of the pool trains.)
    downstream_book_frac: float = 0.35
    ds_val_frac: float  = 0.25
    ds_test_frac: float = 0.30

    # ---------- tokenizer ----------
    vocab_size: int = 32_000
    tokenizer_model: str = "unigram"       # unigram | wordpiece
    min_token_freq: int = 2
    # Grapheme-cluster constraint: forbid vocabulary pieces that BEGIN with a
    # Unicode combining mark. A matra / hasant can then never start a token, so
    # every segmentation is cluster-aligned by construction. Single-codepoint
    # mark pieces are kept but score-penalised, so stray OCR marks degrade to a
    # rare expensive piece instead of <unk>.
    grapheme_safe_vocab: bool = True
    stray_mark_penalty: float = 12.0

    # ---------- base model / vocabulary transfer ----------
    # Chosen empirically by the base-selection stage rather than assumed. Only
    # MLM-head models are eligible: BanglaBERT/BanglishBERT are ELECTRA
    # discriminators (replaced-token detection) and have no MLM head to continue.
    base_candidates: Tuple[str, ...] = (
        "xlm-roberta-base",
        "google/muril-base-cased",
        "sagorsarker/bangla-bert-base",
        "l3cube-pune/bengali-bert",
    )
    base_select_steps: int = 300           # short probe DAPT per candidate
    base_model: str = "xlm-roberta-base"   # overwritten by the selection stage
    # Native vocabulary is the DEFAULT for the main model: swapping the vocabulary
    # AND changing the data order at once would confound the curriculum claim.
    # FOCUS vocabulary transfer is evaluated as its own arm instead.
    use_vocab_transfer: bool = False
    focus_aux_dim: int = 300               # PPMI-SVD (2/3) + char-ngram-SVD (1/3)
    focus_topk: int = 32                   # anchors per non-overlapping token
    focus_temperature: float = 0.1

    # ---------- pretraining ----------
    max_len: int = 256
    mlm_prob: float = 0.15
    whole_word_mask: bool = True
    per_device_bs: int = 32
    grad_accum: int = 4
    lr: float = 5e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    # Steps are DERIVED from target_epochs over the pretraining pool and then
    # capped. The pool is ~28k chunks; a fixed 20k steps would have been ~180
    # epochs, which overfits a corpus this size badly.
    target_epochs: int = 20
    max_steps_cap: int = 8_000
    min_steps: int = 400
    early_stop_patience: int = 4           # evaluations without improvement
    log_every: int = 100
    eval_every: int = 250
    save_every: int = 500

    # ---------- two-phase vocabulary-transfer schedule (fixes FOCUS collapse) --
    # Replacing 97% of the embedding matrix and immediately backpropping into the
    # whole network destroys the FOCUS initialisation. Phase 1 trains embeddings
    # only, with the encoder frozen and a higher LR; phase 2 unfreezes everything.
    embed_warmup_frac: float = 0.15
    embed_warmup_lr: float = 5e-4

    # ---------- GRADE-CL (contribution C1) ----------
    curriculum: str = "grade_cl"           # grade_cl | random | anti | length_cl | no_dapt
    difficulty_weights: Dict[str, float] = field(default_factory=lambda: {
        "grade":      0.50,   # the expert curriculum signal — dominant by design
        "rarity":     0.20,   # corpus-level token rarity
        "sent_len":   0.12,   # mean sentence length
        "ttr":        0.10,   # type-token ratio (lexical diversity)
        "word_len":   0.08,   # mean grapheme-cluster word length
    })
    c0: float = 0.15                       # initial competence (Platanios et al., 2019)
    competence_power: float = 2.0          # p in c(t) = ((1-c0^p)*t/T + c0^p)^(1/p)
    curriculum_frac: float = 0.75          # fraction of training during which competence ramps to 1
    replay_ratio: float = 0.25             # fraction of each batch drawn from already-mastered (easier) data
    difficulty_bins: int = 20              # for the competence CDF lookup

    # ---------- optional extra objectives (OFF by default; each is one more ablation arm) ----------
    use_ocr_weighted_mlm: bool = False     # weight MLM loss by chunk OCR agreement
    ocr_weight_floor: float = 0.5
    use_dual_view_contrastive: bool = False# Surya vs Tesseract views as natural positive pairs
    contrastive_weight: float = 0.1
    contrastive_temp: float = 0.05

    # ---------- downstream ----------
    ft_epochs: int = 4
    ft_lr: float = 2e-5
    ft_bs: int = 32
    ft_max_len: int = 256
    seeds: Tuple[int, ...] = (13, 42, 101, 2024, 31337)
    baselines: Tuple[str, ...] = (
        "xlm-roberta-base",
        "bert-base-multilingual-cased",
        "csebuetnlp/banglabert",
        "google/muril-base-cased",
        "sagorsarker/bangla-bert-base",
    )

    # ---------- statistics ----------
    bootstrap_n: int = 10_000
    alpha: float = 0.05

    def __post_init__(self):
        if self.smoke_test:
            self.target_epochs = 2; self.max_steps_cap = 200; self.min_steps = 50
            self.eval_every = 50; self.save_every = 100; self.log_every = 10
            self.early_stop_patience = 99
            self.ft_epochs = 1
            self.seeds = (13, 42)
            self.vocab_size = 8_000
            self.baselines = ("xlm-roberta-base", "bert-base-multilingual-cased")
            self.base_candidates = ("xlm-roberta-base", "google/muril-base-cased")
            self.base_select_steps = 40
            self.bootstrap_n = 1_000

cfg = CFG()

# ---- auto-discover DATA_ROOT (Kaggle mount paths vary) --------------------
def _find_data_root(given: str) -> str:
    p = Path(given)
    if (p / "metadata" / "books_catalog.csv").exists():
        return str(p)
    for cand in Path("/kaggle/input").rglob("books_catalog.csv"):
        if cand.parent.name == "metadata" and (cand.parent.parent / "class_9_10").exists():
            print(f"[auto] DATA_ROOT resolved to {cand.parent.parent}")
            return str(cand.parent.parent)
    raise FileNotFoundError(
        f"Could not find the dataset. Looked at {given} and scanned /kaggle/input. "
        "Attach the NCTB dataset to the notebook and/or set CFG.data_root."
    )

cfg.data_root = _find_data_root(cfg.data_root)
DATA_ROOT = Path(cfg.data_root)
OUT = Path(cfg.out_dir); OUT.mkdir(parents=True, exist_ok=True)
for sub in ["stages", "artifacts", "models", "figures", "tables", "logs", "cache"]:
    (OUT / sub).mkdir(exist_ok=True)

print(json.dumps({k: (str(v) if isinstance(v, Path) else v)
                  for k, v in asdict(cfg).items()}, indent=2, default=str)[:2000])


In [ ]:
# ============================================================================
# CELL 3 — Stage manager, determinism, timing, logging
# ============================================================================
RUN_T0 = time.time()

def set_all_seeds(s: int):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s); hf_set_seed(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_all_seeds(cfg.seed)

def jdump(obj, **kw) -> str:
    """json.dumps that tolerates numpy scalars/arrays (pandas returns np.int64,
    which the stdlib encoder refuses)."""
    def _d(o):
        if isinstance(o, (np.integer,)):  return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.bool_,)):    return bool(o)
        if isinstance(o, np.ndarray):     return o.tolist()
        if isinstance(o, Path):           return str(o)
        return str(o)
    return json.dumps(obj, default=_d, **kw)

LOG_PATH = OUT / "logs" / "run.log"
def log(msg: str):
    line = f"[{time.strftime('%H:%M:%S')}  +{(time.time()-RUN_T0)/60:6.1f}m] {msg}"
    print(line, flush=True)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")

def budget_left_h() -> float:
    return cfg.session_budget_hours - (time.time() - RUN_T0) / 3600

def budget_exhausted(margin_h: float = 0.25) -> bool:
    return budget_left_h() < margin_h

# ---- stage checkpointing --------------------------------------------------
STAGE_DIR = OUT / "stages"
if cfg.resume_from:
    src = Path(cfg.resume_from)
    if src.exists():
        for sub in ["stages", "artifacts", "models", "cache"]:
            if (src / sub).exists():
                shutil.copytree(src / sub, OUT / sub, dirs_exist_ok=True)
        log(f"resumed prior state from {src}")

def _marker(name: str) -> Path:
    return STAGE_DIR / f"{name}.v{cfg.pipeline_version}.done"

def stage_done(name: str) -> bool:
    if name in cfg.force_stages:
        _marker(name).unlink(missing_ok=True)
        return False
    return _marker(name).exists()

def mark_stage(name: str, meta: Optional[dict] = None):
    _marker(name).write_text(jdump(meta or {"t": time.time()}))
    log(f"stage '{name}' complete")

class stage:
    """with stage('name'): ...   -> skipped entirely if already done."""
    def __init__(self, name): self.name = name; self.skip = stage_done(name)
    def __enter__(self):
        if self.skip: log(f"stage '{self.name}' already done — skipping")
        else: log(f"stage '{self.name}' starting"); self.t0 = time.time()
        return self
    def __exit__(self, exc_type, exc, tb):
        if exc_type is not None: return False
        if not self.skip: mark_stage(self.name, {"seconds": round(time.time()-self.t0, 1)})
        return False

# ---- artifact IO ----------------------------------------------------------
def save_art(obj, name: str):
    p = OUT / "artifacts" / name
    if name.endswith(".parquet"): obj.to_parquet(p, index=False)
    elif name.endswith(".csv"):   obj.to_csv(p, index=False)
    elif name.endswith(".json"):  p.write_text(jdump(obj, ensure_ascii=False, indent=2))
    elif name.endswith(".npy"):   np.save(p, obj)
    else: raise ValueError(name)
    return p

def load_art(name: str):
    p = OUT / "artifacts" / name
    if not p.exists(): return None
    if name.endswith(".parquet"): return pd.read_parquet(p)
    if name.endswith(".csv"):     return pd.read_csv(p)
    if name.endswith(".json"):    return json.loads(p.read_text())
    if name.endswith(".npy"):     return np.load(p, allow_pickle=True)

log(f"stage manager ready — out_dir={OUT}")


---
## Section 1 — Corpus construction

We load all 156 books (46,560 chunks) from the five class bands, attach the book-level catalog
metadata (including `mean_agreement`, `source_type`, `pages`), then apply a conservative cleaning
pass. Cleaning is *conservative on purpose*: OCR noise is a property of the domain we want the model
to be robust to, so we remove only structurally broken records (front-matter, near-duplicates,
degenerate length, script/language mismatch) and keep noisy-but-real text.


In [ ]:
# ============================================================================
# CELL 4 — Load catalog + all chunk records
# ============================================================================
BANDS = ["class_1_4", "class_5_6", "class_7_8", "class_9_10", "class_11_12"]

def load_catalog() -> pd.DataFrame:
    cat = pd.read_csv(DATA_ROOT / "metadata" / "books_catalog.csv")
    cat["class_band"] = cat["class_band"].astype(str)
    return cat

def iter_jsonl_files():
    for band in BANDS:
        dd = DATA_ROOT / band / "data"
        if not dd.exists(): continue
        for p in sorted(dd.rglob("*.jsonl")):
            yield band, p

def load_chunks() -> pd.DataFrame:
    rows, bad = [], 0
    files = list(iter_jsonl_files())
    for band, p in tqdm(files, desc="reading jsonl"):
        book_file = p.stem                       # e.g. biology_ben
        with open(p, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try: r = json.loads(line)
                except Exception: bad += 1; continue
                r["class_band"] = band.replace("class_", "").replace("_", "-")
                r["book_file"]  = book_file
                r["grade_dir"]  = p.parent.name if p.parent.name != "data" else band
                rows.append(r)
    df = pd.DataFrame(rows)
    if bad: log(f"skipped {bad} malformed jsonl lines")
    return df

with stage("load"):
    if not stage_done("load"):
        catalog = load_catalog()
        df = load_chunks()
        # --- canonical book id: the unit of our leakage-safe splits -------------
        # MUST include grade_dir. In class_1_4 / class_5_6 / class_7_8 the same
        # file stem recurs under class_1/, class_2/, ... (e.g. amar_bangla_boi_ben
        # exists for several grades), so class_band|stem silently merged distinct
        # physical textbooks -- 156 books collapsed to 114, and a "book" then
        # spanned multiple grades, contaminating the grade signal AND the splits.
        df["book_id"] = df["class_band"] + "|" + df["grade_dir"] + "|" + df["book_file"]
        # numeric grade for the curriculum signal: use the *lower* grade of a shared band
        df["grade_lo"] = df["class"].astype(str).str.split("-").str[0].astype(int)
        df["grade_hi"] = df["class"].astype(str).str.split("-").str[-1].astype(int)
        df["grade"]    = (df["grade_lo"] + df["grade_hi"]) / 2.0
        for c in ["ocr_agreement", "n_chars", "chapter_no", "chunk_index"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df["text"] = df["text"].fillna("").astype(str)
        save_art(catalog, "catalog.parquet")
        save_art(df, "chunks_raw.parquet")

catalog = load_art("catalog.parquet")
df_raw  = load_art("chunks_raw.parquet")
n_books = df_raw.book_id.nunique()
log(f"catalog: {len(catalog)} books | chunks: {len(df_raw):,} | books present: {n_books}")
if n_books != len(catalog):
    log(f"WARNING: {n_books} distinct book_ids vs {len(catalog)} catalog rows — "
        "check for filename collisions across grade folders")
display(df_raw[["chunk_id","class_band","subject","language","grade","n_chars","ocr_agreement"]].head(3))


In [ ]:
# ============================================================================
# CELL 5 — Cleaning, normalisation, near-duplicate removal
# ============================================================================
# Bengali-aware normalisation: NFC, canonical nukta/ya-phala forms, zero-width cleanup.
_ZW = re.compile(r"[​‌‍﻿]")          # ZWSP/ZWNJ/ZWJ/BOM
_MULTISPACE = re.compile(r"[ \t ]+")
_MULTINL = re.compile(r"\n{3,}")
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
# OCR junk: long runs of isolated latin single letters inside Bengali, repeated punctuation
_JUNK_RUN = re.compile(r"(?:\b[A-Za-z]\b[\s\-_.]{0,2}){6,}")
_PUNCT_RUN = re.compile(r"([^\w\sঀ-৿])\1{3,}")
_GRAPHEME = re.compile(r"\X")
_BN = re.compile(r"[ঀ-৿]")
_LATIN = re.compile(r"[A-Za-z]")
_SENT_SPLIT = re.compile(r"[।!?\.\n]+")

# --- Bangla orthographic normalisation ---------------------------------------
# Bengali admits several byte sequences for the same rendered grapheme, and OCR
# engines are inconsistent about which they emit. Left unnormalised, the model
# sees ড় and ড+় as unrelated types and the vocabulary is silently split.
#
#  * Nukta letters (U+09DC ড়, U+09DD ঢ়, U+09DF য়) are Unicode *composition
#    exclusions*: NFC will NOT recompose them, so both forms survive normalisation.
#    We map them to one canonical decomposed form explicitly.
#  * Two-part vowel signs (ো = ে+া, ৌ = ে+ৗ) ARE canonically composable, so NFC
#    repairs the split forms OCR produces.
#  * NFKC additionally folds compatibility variants (full-width digits, ligature
#    presentation forms) that scanned material picks up.
_NUKTA_MAP = {"\u09dc": "\u09a1\u09bc", "\u09dd": "\u09a2\u09bc", "\u09df": "\u09af\u09bc",
              "\u09b0\u09bc": "\u09b0"}
_BN_DIGITS = {d: str(i) for i, d in enumerate("০১২৩৪৫৬৭৮৯")}

def normalize_text(t: str) -> str:
    t = unicodedata.normalize(cfg.unicode_form, t)
    t = _CTRL.sub(" ", t)
    t = _ZW.sub("", t)
    if cfg.normalize_nukta:
        for a, b in _NUKTA_MAP.items(): t = t.replace(a, b)
    if cfg.normalize_bn_digits:
        for a, b in _BN_DIGITS.items(): t = t.replace(a, b)
    t = _JUNK_RUN.sub(" ", t)
    t = _PUNCT_RUN.sub(r"\1", t)
    t = _MULTISPACE.sub(" ", t)
    t = _MULTINL.sub("\n\n", t)
    return t.strip()

def strip_furniture(frame: pd.DataFrame, min_frac: float) -> Tuple[pd.DataFrame, dict]:
    """
    Remove per-page furniture: running headers/footers that OCR copies into every
    chunk of a book (book title, "শিক্ষাবর্ষ ২০২৬", board imprint, page banners).

    These are the single largest source of near-duplicate n-grams in a scanned
    textbook corpus. They inflate every similarity metric, dominate the frequency
    tail the tokenizer trains on, and give the model an easy, meaningless
    prediction target. Detection is per BOOK, so genuinely repeated subject
    vocabulary across books is untouched.
    """
    report, out = {}, []
    for book, g in frame.groupby("book_id"):
        n = len(g)
        if n < 8: out.append(g); continue
        counts = Counter()
        for t in g["text"]:
            words = t.split()
            for k in (4, 5, 6, 7):                    # candidate furniture lengths
                for start in (0, max(0, len(words) - k)):   # head and tail only
                    ng = " ".join(words[start:start + k])
                    if len(ng) > 8: counts[ng] += 1
        furniture = {ng for ng, c in counts.items() if c >= max(3, int(min_frac * n))}
        # keep only maximal strings so we do not strip a fragment twice
        furniture = {f for f in furniture
                     if not any(f != o and f in o for o in furniture)}
        if furniture:
            pat = re.compile("|".join(sorted((re.escape(f) for f in furniture),
                                             key=len, reverse=True)))
            g = g.assign(text=[pat.sub(" ", t) for t in g["text"]])
            report[book] = len(furniture)
        out.append(g)
    return pd.concat(out).sort_index(), report

def lexicon_coverage(frame: pd.DataFrame, min_freq: int) -> np.ndarray:
    """
    Self-supervised OCR-quality score: the fraction of a chunk's word tokens that
    occur at least `min_freq` times ACROSS THE CORPUS.

    Garbled OCR generates strings that appear once and never again ("মিশ্রীয়",
    stray latin runs, broken conjuncts), so a low coverage score is a direct
    proxy for recognition failure. Building the lexicon from the corpus itself
    avoids depending on an external Bengali wordlist, which Kaggle may not have
    offline and which would not cover textbook technical vocabulary anyway.
    """
    freq = Counter()
    for t in frame["text"]: freq.update(t.split())
    good = {w for w, c in freq.items() if c >= min_freq}
    cov = np.empty(len(frame), dtype=np.float32)
    for i, t in enumerate(frame["text"]):
        w = t.split()
        cov[i] = sum(1 for x in w if x in good) / max(len(w), 1)
    return cov

def script_ratio(t: str) -> Tuple[float, float]:
    n = max(len(t), 1)
    return len(_BN.findall(t)) / n, len(_LATIN.findall(t)) / n

def minhash_dedup(texts: List[str], threshold: float, num_perm: int = 64,
                  k: int = 5, seed: int = 0) -> np.ndarray:
    """
    Self-contained MinHash + banded LSH near-duplicate detection (no third-party
    dependency).  Character k-gram shingles -> num_perm min-hashes -> b bands of
    r rows, with (b, r) chosen so the LSH S-curve inflects nearest `threshold`
    ((1/b)^(1/r) ~= threshold).  Returns a boolean keep-mask; the first member of
    each near-duplicate cluster is kept.
    """
    rng = np.random.default_rng(seed)
    A = rng.integers(1, 2**61, size=num_perm, dtype=np.uint64)
    B = rng.integers(0, 2**61, size=num_perm, dtype=np.uint64)

    best = None
    for r in range(1, num_perm + 1):
        if num_perm % r: continue
        b = num_perm // r
        t = (1.0 / b) ** (1.0 / r)
        if best is None or abs(t - threshold) < abs(best[2] - threshold):
            best = (r, b, t)
    r, nb, t_eff = best

    seen = [dict() for _ in range(nb)]
    keep = np.ones(len(texts), dtype=bool)
    for i, txt in enumerate(tqdm(texts, desc=f"dedup (LSH {nb}x{r}, t≈{t_eff:.2f})")):
        s = txt[:4000]
        if len(s) < k:
            keep[i] = False; continue
        shingles = {zlib.crc32(s[j:j+k].encode("utf8")) for j in range(len(s) - k + 1)}
        if not shingles:
            keep[i] = False; continue
        x = np.fromiter(shingles, dtype=np.uint64, count=len(shingles))
        # (a*x + b) with uint64 wraparound is a fine universal-ish hash family here
        sig = (A[:, None] * x[None, :] + B[:, None]).min(axis=1)
        bands = [sig[j*r:(j+1)*r].tobytes() for j in range(nb)]
        if any(bk in seen[j] for j, bk in enumerate(bands)):
            keep[i] = False
        else:
            for j, bk in enumerate(bands): seen[j][bk] = i
    return keep

with stage("clean"):
    if not stage_done("clean"):
        d = df_raw.copy()
        n0 = len(d)
        d["text"] = [normalize_text(t) for t in tqdm(d["text"].tolist(), desc="normalise")]
        d["n_chars"] = d["text"].str.len()
        sr = [script_ratio(t) for t in d["text"]]
        d["bn_ratio"]    = [x[0] for x in sr]
        d["latin_ratio"] = [x[1] for x in sr]

        rules = {}   # NOTE: cast every count to a python int -- pandas hands back
                     # numpy.int64, which json.dumps refuses to serialise.
        m = d["n_chars"].between(cfg.min_chars, cfg.max_chars);          rules["length"] = int((~m).sum()); d = d[m]
        # language/script consistency: a 'bn' chunk with <15% Bengali glyphs is an OCR failure
        m = ~(((d.language == "bn") & (d.bn_ratio < 0.15)) |
              ((d.language == "en") & (d.latin_ratio < 0.15)));          rules["script_mismatch"] = int((~m).sum()); d = d[m]
        if cfg.drop_frontmatter_chapters:
            m = ~((d.chapter_no == 1) & (d.chunk_index == 1));           rules["frontmatter"] = int((~m).sum()); d = d[m]
        if cfg.min_ocr_agreement > 0:
            m = d["ocr_agreement"].fillna(1.0) >= cfg.min_ocr_agreement; rules["ocr_floor"] = int((~m).sum()); d = d[m]

        # --- book-level quality gate from the dataset's own QC columns ---------
        if "recon_ratio" in catalog.columns:
            bad_books = set(catalog.loc[
                (catalog["recon_ratio"] - 1.0).abs() > cfg.max_recon_deviation, "key"])
            if bad_books:
                m = ~d["book_file"].str.lower().isin({b.lower() for b in bad_books})
                rules["recon_ratio_book"] = int((~m).sum())
                d = d[m]

        # --- running headers / footers ----------------------------------------
        if cfg.strip_repeated_furniture:
            d, furn = strip_furniture(d, cfg.furniture_min_book_frac)
            d["text"] = d["text"].str.replace(r"\s+", " ", regex=True).str.strip()
            d["n_chars"] = d["text"].str.len()
            m = d["n_chars"] >= cfg.min_chars
            rules["furniture_emptied"] = int((~m).sum()); d = d[m]
            log(f"furniture: stripped repeated head/tail n-grams from {len(furn)} books")

        # --- OCR quality gate --------------------------------------------------
        d = d.reset_index(drop=True)
        d["lex_coverage"] = lexicon_coverage(d, cfg.lex_min_freq)
        m = d["lex_coverage"] >= cfg.lex_coverage_min
        rules["low_lex_coverage"] = int((~m).sum())
        log(f"lexicon coverage: median={d.lex_coverage.median():.3f}, "
            f"dropping {int((~m).sum()):,} chunks below {cfg.lex_coverage_min}")
        d = d[m]

        d = d.reset_index(drop=True)
        keep = minhash_dedup(d["text"].tolist(), cfg.dedup_threshold)
        rules["near_duplicate"] = int((~keep).sum())
        d = d[keep].reset_index(drop=True)

        log(f"cleaning: {n0:,} -> {len(d):,} chunks  ({100*len(d)/n0:.1f}% kept)")
        log("removed by rule: " + jdump(rules))
        save_art(d, "chunks_clean.parquet")
        save_art({"n_before": n0, "n_after": int(len(d)), "removed": {k: int(v) for k, v in rules.items()}},
                 "cleaning_report.json")

df = load_art("chunks_clean.parquet")
clean_report = load_art("cleaning_report.json")
log(f"clean corpus: {len(df):,} chunks · {df.book_id.nunique()} books · {df.text.str.len().sum()/1e6:.1f}M chars")


In [ ]:
# ============================================================================
# CELL 6 — Publication figure style (colour-blind-safe, validated palette)
# ============================================================================
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Categorical slots assigned in FIXED order, never cycled. Slots 1-3 are all-pairs
# CVD-safe; we cap any single axes at 4 series and facet beyond that.
PAL   = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQ   = ["#d3e3f7", "#a7c6ee", "#7aa8e4", "#4d8bdb", "#2a78d6", "#1f5aa0", "#153c6b"]  # one hue, light->dark
INK   = "#0b0b0b"; INK2 = "#52514e"; MUTED = "#8a8880"; SURFACE = "#ffffff"; GRID = "#e6e5e1"

mpl.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 330, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.bbox": "tight", "savefig.facecolor": SURFACE,
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.edgecolor": GRID, "axes.linewidth": 0.8, "axes.labelcolor": INK2,
    "axes.titlesize": 10.5, "axes.titleweight": "600", "axes.titlecolor": INK, "axes.titlelocation": "left",
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.7,
    "xtick.color": INK2, "ytick.color": INK2, "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "legend.frameon": False, "legend.fontsize": 8.5, "legend.labelcolor": INK2,
    "lines.linewidth": 2.0, "lines.markersize": 5,
})

def finish(ax, title=None, sub=None, xlabel=None, ylabel=None, ygrid=True, xgrid=False):
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    ax.spines["left"].set_color(GRID); ax.spines["bottom"].set_color(GRID)
    ax.grid(axis="y", visible=ygrid); ax.grid(axis="x", visible=xgrid)
    if title: ax.set_title(title, pad=14 if sub else 8)
    if sub:   ax.text(0, 1.02, sub, transform=ax.transAxes, fontsize=8.5, color=MUTED, va="bottom")
    ax.set_xlabel(xlabel or "", color=INK2); ax.set_ylabel(ylabel or "", color=INK2)
    return ax

def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(OUT / "figures" / f"{name}.{ext}")
    log(f"figure saved: {name}")

print("figure style ready")


In [ ]:
# ============================================================================
# CELL 7 — Figure 1: corpus composition
# ============================================================================
band_order = ["1-4", "5-6", "7-8", "9-10", "11-12"]
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.3))

# (a) chunks per band, split by language — 2 series, direct-labelled
ax = axes[0]
piv = (df.groupby(["class_band", "language"]).size().unstack(fill_value=0).reindex(band_order))
x = np.arange(len(band_order)); w = 0.38
for i, lang in enumerate(["bn", "en"]):
    v = piv.get(lang, pd.Series(0, index=band_order)).values
    ax.bar(x + (i - 0.5) * (w + 0.02), v, w, color=PAL[i], label={"bn":"Bengali","en":"English"}[lang])
    for xi, vi in zip(x + (i - 0.5) * (w + 0.02), v):
        ax.text(xi, vi, f"{vi/1000:.1f}k", ha="center", va="bottom", fontsize=7.5, color=INK2)
ax.set_xticks(x); ax.set_xticklabels(band_order); ax.legend(loc="upper left", ncols=2)
finish(ax, "Chunks per class band", "after cleaning", "class band", "chunks")

# (b) OCR agreement distribution by band — single hue sequential (magnitude by band)
ax = axes[1]
for i, b in enumerate(band_order):
    v = df.loc[df.class_band == b, "ocr_agreement"].dropna().values
    if len(v) == 0: continue
    xs = np.linspace(0, 1, 200)
    kd = sp_stats.gaussian_kde(v) if len(v) > 50 else None
    ys = kd(xs) if kd else np.histogram(v, bins=200, range=(0,1), density=True)[0]
    ax.plot(xs, ys / max(ys.max(), 1e-9), color=SEQ[min(i+1, len(SEQ)-1)], lw=1.8, label=b)
ax.legend(title="band", ncols=2, loc="upper center")
finish(ax, "OCR dual-engine agreement", "Surya vs Tesseract, density (scaled)", "agreement", "density")

# (c) chunk length distribution
ax = axes[2]
ax.hist(df["n_chars"].clip(0, 4000), bins=60, color=PAL[0], edgecolor=SURFACE, linewidth=0.5)
med = df["n_chars"].median()
ax.axvline(med, color=INK2, lw=1.2, ls="--")
ax.text(med, ax.get_ylim()[1]*0.92, f" median {med:.0f}", color=INK2, fontsize=8)
finish(ax, "Chunk length", "characters (clipped at 4,000)", "characters", "chunks")

fig.tight_layout(); savefig(fig, "fig1_corpus_composition"); plt.show()

summary = (df.groupby("class_band")
             .agg(books=("book_id","nunique"), chunks=("chunk_id","size"),
                  chars=("n_chars","sum"), mean_agree=("ocr_agreement","mean"))
             .reindex(band_order))
display(summary)
save_art(summary.reset_index(), "corpus_summary.csv")


---
## Section 2 — Leakage-safe, **book-level** splits (contribution C4)

Adjacent chunks of one chapter overlap heavily in vocabulary, named entities and even sentences
(OCR page boundaries duplicate headers/footers). A random chunk-level split therefore puts
near-duplicates on both sides of the train/test line and inflates every downstream number.

We split at the **book** level with stratification over (class band × language × subject family):

* **Pretraining pool** — 70 % of books. Only these are seen during DAPT.
* **Downstream pool** — 30 % of books, *never seen during pretraining*, further split
  into train / val / test **again by book**.

We additionally report a chunk-level-split control so the paper can quantify the inflation.


In [ ]:
# ============================================================================
# CELL 8 — Book-level splits
# ============================================================================
def allocate_books(book_df: pd.DataFrame, fracs: Dict[str, float], seed: int,
                   group_col: str = "class_band", balance_col: str = "language"
                   ) -> Dict[str, str]:
    """
    Assign whole books to splits with a per-class-band quota, using the
    largest-remainder method.

    Why not the previous nested stratified sampler: with only 4 books in the
    11-12 band, stratifying on class_band x language produced strata of 2 books,
    so that band could never reach all four splits. The 5-class grade task then
    contained a test class with zero training examples -- untrainable by
    construction, and it silently capped macro-F1 at 0.8 for every model.

    Guarantee here: any band with at least len(fracs) books contributes at least
    one book to EVERY split. `balance_col` is interleaved within a band so the
    two languages spread across splits rather than clustering.
    """
    rng = np.random.default_rng(seed)
    names = list(fracs); weights = np.array([fracs[n] for n in names], dtype=float)
    weights = weights / weights.sum()
    assign: Dict[str, str] = {}

    for band, grp in book_df.groupby(group_col):
        # interleave languages so a band's books alternate bn/en down the list
        buckets = []
        for _, sub in grp.groupby(balance_col):
            ids = sub["book_id"].tolist(); rng.shuffle(ids); buckets.append(ids)
        ordered = [b.pop(0) for _ in range(max(map(len, buckets), default=0))
                   for b in buckets if b]
        n = len(ordered)
        if n == 0: continue

        quota = weights * n
        base = np.floor(quota).astype(int)
        # largest remainder distributes the leftovers
        for j in np.argsort(-(quota - base))[: n - base.sum()]:
            base[j] += 1
        # every split gets >=1 book when the band is big enough to allow it
        if n >= len(names):
            while (base == 0).any():
                base[int(np.argmax(base))] -= 1
                base[int(np.argmin(base))] += 1

        pos = 0
        for name, cnt in zip(names, base):
            for bid in ordered[pos:pos + cnt]: assign[bid] = name
            pos += cnt
    return assign

with stage("split"):
    if not stage_done("split"):
        books = (df.groupby("book_id")
                   .agg(class_band=("class_band","first"), language=("language","first"),
                        subject=("subject","first"), chunks=("chunk_id","size"),
                        grade=("grade","mean"))
                   .reset_index())
        books["subject_fam"] = books["subject"].str.lower().str.replace(r"[^a-z]", "", regex=True).str[:4]

        # One allocation over all four splits at once, quota'd per class band.
        p = cfg.downstream_book_frac
        FRACS = {
            "pretrain": 1.0 - p,
            "ds_train": p * (1.0 - cfg.ds_val_frac - cfg.ds_test_frac),
            "ds_val":   p * cfg.ds_val_frac,
            "ds_test":  p * cfg.ds_test_frac,
        }
        assert FRACS["ds_train"] > 0, (
            "ds_val_frac + ds_test_frac must be < 1.0 (they are fractions of the "
            f"downstream pool); got {cfg.ds_val_frac} + {cfg.ds_test_frac}")
        book2split = allocate_books(books, FRACS, cfg.seed)
        df["split"] = df["book_id"].map(book2split).fillna("pretrain")

        # chunk-level control split (for the leakage-inflation experiment)
        rng = np.random.default_rng(cfg.seed)
        perm = rng.permutation(len(df))
        ctrl = np.array(["ds_train"] * len(df), dtype=object)
        ctrl[perm[: int(0.15*len(df))]] = "ds_val"
        ctrl[perm[int(0.15*len(df)) : int(0.30*len(df))]] = "ds_test"
        df["split_chunklevel"] = ctrl

        save_art(df, "chunks_split.parquet")
        save_art(books, "books.parquet")

df    = load_art("chunks_split.parquet")
books = load_art("books.parquet")
SPLITS = ["pretrain", "ds_train", "ds_val", "ds_test"]
display(pd.crosstab(df["split"], df["class_band"]).reindex(SPLITS).fillna(0).astype(int))
log("books per split: " + jdump(df.groupby("split").book_id.nunique().to_dict()))

# --- hard invariants. A silently empty split is the single most expensive
# --- failure mode here: everything downstream still "runs" and produces nothing.
assert set(df[df.split == "pretrain"].book_id) & set(df[df.split.str.startswith("ds_")].book_id) == set(), \
    "book leakage between pretraining and downstream pools"
_empty = [s for s in SPLITS if (df.split == s).sum() == 0]
assert not _empty, (
    f"empty split(s): {_empty}. Check CFG.downstream_book_frac / ds_val_frac / ds_test_frac — "
    f"ds_val_frac + ds_test_frac must be < 1.0 (they are fractions OF THE DOWNSTREAM POOL). "
    f"Currently {cfg.ds_val_frac} + {cfg.ds_test_frac} = {cfg.ds_val_frac + cfg.ds_test_frac}.")
# every class band must reach every downstream split, or the grade task contains
# a class that is untrainable by construction
_cov = pd.crosstab(df["class_band"], df["split"])
_gap = [(b, s) for b in _cov.index for s in ["ds_train", "ds_val", "ds_test"]
        if _cov.loc[b, s] == 0]
if _gap:
    log(f"WARNING: class band(s) missing from a downstream split: {_gap}. "
        "Bands with fewer books than splits cannot be covered; T1 will drop them.")
log("split assertions passed: pools disjoint, all four splits non-empty")


---
## Section 3 — The GRADE difficulty index (contributions C1 + C2)

For chunk $x$ from a book taught at grade $g(x)\in[1,12]$ we define

$$
D(x)\;=\;w_g\,\tilde g(x)\;+\;\sum_{f\in\mathcal F} w_f\,\Phi\!\big(z_f(x)\big),
\qquad \mathcal F=\{\text{rarity},\text{sent\_len},\text{ttr},\text{word\_len}\}
$$

where $\tilde g = (g-1)/11$ is the **expert curriculum signal**, $z_f$ is the within-language
z-score of intrinsic feature $f$, and $\Phi$ is the standard-normal CDF (squashing to $[0,1]$ so
no single outlier feature dominates). Features are computed **on the pretraining pool only** to
avoid test-time information leaking into the curriculum.

Two things make this more than a heuristic:

1. **$w_g$ dominates (0.50).** The ordering is anchored on human expert judgement, not on a proxy.
2. **We can validate the proxies.** Because $g$ is observed, we report Spearman $\rho(f, g)$ for
   every intrinsic feature — i.e. *how well the proxies used by all prior curriculum-learning work
   recover expert-assigned difficulty*. This is contribution **C2** and yields Figure 2.


In [ ]:
# ============================================================================
# CELL 9 — Difficulty features + GRADE index + proxy validation (Figure 2)
# ============================================================================
def graphemes(w: str) -> int:
    return len(_GRAPHEME.findall(w))

def text_features(t: str, freq: Counter, total: float) -> Dict[str, float]:
    words = t.split()
    n = max(len(words), 1)
    sents = [s for s in _SENT_SPLIT.split(t) if s.strip()]
    types = len(set(words))
    # rarity: mean negative log unigram probability (higher = rarer vocabulary)
    rar = float(np.mean([-math.log((freq.get(w, 0) + 1) / total) for w in words])) if words else 0.0
    return {
        "rarity":   rar,
        "sent_len": n / max(len(sents), 1),
        "ttr":      types / n,
        "word_len": float(np.mean([graphemes(w) for w in words])) if words else 0.0,
    }

with stage("difficulty"):
    if not stage_done("difficulty"):
        pre = df[df.split == "pretrain"]
        freq_by_lang, total_by_lang = {}, {}
        for lang, g in pre.groupby("language"):
            c = Counter()
            for t in g["text"]: c.update(t.split())
            freq_by_lang[lang] = c; total_by_lang[lang] = max(sum(c.values()), 1)

        feats = []
        for lang, g in df.groupby("language"):
            f = freq_by_lang.get(lang, Counter()); tot = total_by_lang.get(lang, 1)
            sub = pd.DataFrame([text_features(t, f, tot) for t in tqdm(g["text"], desc=f"feats[{lang}]")],
                               index=g.index)
            feats.append(sub)
        FEAT = pd.concat(feats).sort_index()
        df = pd.concat([df, FEAT], axis=1)

        # within-language z-score then normal CDF squash
        for f in ["rarity", "sent_len", "ttr", "word_len"]:
            z = df.groupby("language")[f].transform(lambda s: (s - s.mean()) / (s.std() + 1e-9))
            df[f"n_{f}"] = sp_stats.norm.cdf(z.clip(-4, 4))
        df["n_grade"] = (df["grade"] - 1) / 11.0

        W = cfg.difficulty_weights
        df["difficulty"] = sum(W[k] * df[f"n_{k}"] for k in W)
        df["difficulty"] = (df["difficulty"] - df["difficulty"].min()) / \
                           (df["difficulty"].max() - df["difficulty"].min() + 1e-9)
        # classic length-only proxy curriculum (ablation arm `length_cl`)
        df["difficulty_len"] = df.groupby("language")["n_chars"].rank(pct=True)

        # ---- C2: validate intrinsic proxies against the expert grade signal ----
        val = []
        for f in ["rarity", "sent_len", "ttr", "word_len", "n_chars", "ocr_agreement"]:
            for lang, g in df.groupby("language"):
                gg = g[[f, "grade"]].dropna()
                if len(gg) < 100: continue
                rho, p = sp_stats.spearmanr(gg[f], gg["grade"])
                val.append({"feature": f, "language": lang, "spearman_rho": rho,
                            "p_value": p, "n": len(gg)})
        proxy_val = pd.DataFrame(val)
        save_art(df, "chunks_difficulty.parquet")
        save_art(proxy_val, "proxy_validation.csv")

df         = load_art("chunks_difficulty.parquet")
proxy_val  = load_art("proxy_validation.csv")
display(proxy_val.pivot(index="feature", columns="language", values="spearman_rho").round(3))


In [ ]:
# ============================================================================
# CELL 10 — Figure 2: difficulty index & proxy validation
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.4))

# (a) difficulty distribution by class band — sequential single hue = magnitude
ax = axes[0]
for i, b in enumerate(band_order):
    v = df.loc[df.class_band == b, "difficulty"].values
    if len(v) < 30: continue
    xs = np.linspace(0, 1, 250); ys = sp_stats.gaussian_kde(v)(xs)
    ax.plot(xs, ys, color=SEQ[min(i+1, len(SEQ)-1)], lw=1.9)
    ax.text(xs[np.argmax(ys)], ys.max()*1.02, b, color=SEQ[min(i+1, len(SEQ)-1)],
            fontsize=8, ha="center", weight="600")
finish(ax, "GRADE difficulty index D(x)", "kernel density, direct-labelled by class band",
       "difficulty", "density")

# (b) proxy validation: |Spearman rho| vs the expert grade label
ax = axes[1]
pv = (proxy_val.groupby("feature")["spearman_rho"].mean().sort_values())
cols = [PAL[0] if abs(v) >= 0.2 else MUTED for v in pv.values]
ax.barh(np.arange(len(pv)), pv.values, color=cols, height=0.62)
ax.set_yticks(np.arange(len(pv))); ax.set_yticklabels(pv.index, fontsize=8.5)
for i, v in enumerate(pv.values):
    ax.text(v + (0.01 if v >= 0 else -0.01), i, f"{v:+.2f}", va="center",
            ha="left" if v >= 0 else "right", fontsize=7.5, color=INK2)
ax.axvline(0, color=INK2, lw=0.8)
finish(ax, "Do intrinsic proxies recover expert difficulty?",
       "Spearman ρ against curriculum grade (mean over languages)", "ρ", None, ygrid=False, xgrid=True)

# (c) mean difficulty per grade — the monotonicity check
ax = axes[2]
gm = df.groupby("grade")["difficulty"].agg(["mean", "std", "count"]).reset_index()
ax.plot(gm["grade"], gm["mean"], color=PAL[0], marker="o", zorder=3)
ax.fill_between(gm["grade"], gm["mean"] - gm["std"], gm["mean"] + gm["std"],
                color=PAL[0], alpha=0.14, lw=0)
rho_all, p_all = sp_stats.spearmanr(df["grade"], df["difficulty"])
ax.text(0.03, 0.93, f"ρ(grade, D) = {rho_all:.3f}\np = {p_all:.1e}", transform=ax.transAxes,
        fontsize=8.5, color=INK2, va="top")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
finish(ax, "Difficulty is monotone in grade", "mean ± 1 s.d.", "curriculum grade", "D(x)")

fig.tight_layout(); savefig(fig, "fig2_difficulty_index"); plt.show()
save_art({"spearman_grade_difficulty": float(rho_all), "p": float(p_all)}, "difficulty_validation.json")


---
## Section 4 — A Bengali-aware tokenizer for NCTB (contribution C3, part 1)

XLM-R's 250k SentencePiece vocabulary spends most of its capacity on languages we do not need and
still over-fragments Bengali conjuncts. We train a 32k Unigram vocabulary **on the pretraining pool
only** with a normaliser that:

* applies NFC and strips zero-width joiners that OCR inserts inconsistently,
* enforces a **grapheme-cluster constraint on the vocabulary itself**: no piece may begin with a
  Unicode combining mark, so a matra or hasant can only ever appear attached to its base character
  inside the same piece, and every possible segmentation is cluster-aligned by construction.
  (A Metaspace pre-tokeniser alone does *not* give this — it protects word boundaries only, and
  Unigram will happily emit `('▁আঁক', 'ি')`. We report the measured *cluster-split rate* rather
  than asserting the property.)
* keeps the Latin script intact so the bilingual (ben/eng) corpus shares one vocabulary.

We then measure **fertility** (subword tokens per whitespace word) and **compression** against five
public baselines — the headline efficiency result.


In [ ]:
# ============================================================================
# CELL 11 — Train the NCTB tokenizer
# ============================================================================
TOK_DIR = OUT / "models" / "nctb_tokenizer"

from tokenizers import Regex as TkRegex

def bengali_safe_normalizer():
    return normalizers.Sequence([
        normalizers.NFC(),
        normalizers.Replace(TkRegex(r"[​‌‍﻿]"), ""),  # ZWSP/ZWNJ/ZWJ/BOM
        normalizers.Replace(TkRegex(r"\s+"), " "),
    ])

_MARK_INITIAL = re.compile(r"^\p{M}")     # Mn / Mc / Me: matras, hasant, nukta, ...

def enforce_grapheme_safe_vocab(tok: Tokenizer, specials: List[str], target: int) -> Tokenizer:
    """
    Rebuild a trained Unigram model so that segmentation cannot cut a Bengali
    grapheme cluster.

    Metaspace only protects WORD boundaries -- it happily let Unigram emit
    ('▁আঁক', 'ি'), stranding the ি matra from its base consonant. Bengali
    orthography treats consonant+hasant+consonant+matra as one unit, and
    splitting it produces pieces that correspond to nothing in the language.

    Constraint: no vocabulary piece may BEGIN with a combining mark. A mark can
    then only ever appear attached to a preceding base character inside the same
    piece, which makes every possible segmentation cluster-aligned. Single-mark
    pieces are retained but heavily score-penalised, so genuinely stray marks
    (OCR noise) still tokenise rather than becoming <unk>.
    """
    state = json.loads(tok.to_str())
    vocab = state["model"]["vocab"]           # list of [piece, log_prob]
    kept, dropped, penalised = [], 0, 0
    for piece, score in vocab:
        if piece in specials:
            kept.append([piece, score]); continue
        body = piece[1:] if piece.startswith("▁") else piece
        if body and _MARK_INITIAL.match(body):
            if len(body) == 1:
                kept.append([piece, score - cfg.stray_mark_penalty]); penalised += 1
            else:
                dropped += 1
        else:
            kept.append([piece, score])
    # Trim back to the exact target size. Specials keep their leading positions
    # (so unk_id stays valid) and every single-character piece is retained
    # unconditionally, otherwise pruning would open <unk> holes in the alphabet.
    head = [p for p in kept if p[0] in specials]
    body_pieces = [p for p in kept if p[0] not in specials]
    atoms = [p for p in body_pieces if len(p[0].lstrip("▁")) <= 1]
    multi = sorted((p for p in body_pieces if len(p[0].lstrip("▁")) > 1),
                   key=lambda p: -p[1])
    room = max(0, target - len(head) - len(atoms))
    final = head + atoms + multi[:room]
    state["model"]["vocab"] = final
    log(f"grapheme-safe vocab: dropped {dropped} mark-initial pieces, "
        f"penalised {penalised} stray-mark pieces, final |V| = {len(final):,}")
    return Tokenizer.from_str(json.dumps(state))

def train_tokenizer(texts: List[str], vocab_size: int, kind: str) -> PreTrainedTokenizerFast:
    specials = ["<pad>", "<unk>", "<s>", "</s>", "<mask>"]
    # train oversized when we are going to prune mark-initial pieces afterwards,
    # so the pruned vocabulary still lands on the requested size
    train_v = int(vocab_size * 1.35) if (cfg.grapheme_safe_vocab and kind == "unigram") else vocab_size
    if kind == "unigram":
        tok = Tokenizer(models.Unigram())
        trainer = trainers.UnigramTrainer(
            vocab_size=train_v, special_tokens=specials, unk_token="<unk>",
            shrinking_factor=0.75, max_piece_length=16, n_sub_iterations=2,
        )
    else:
        tok = Tokenizer(models.WordPiece(unk_token="<unk>"))
        trainer = trainers.WordPieceTrainer(
            vocab_size=vocab_size, special_tokens=specials,
            min_frequency=cfg.min_token_freq, continuing_subword_prefix="##",
        )
    tok.normalizer = bengali_safe_normalizer()
    # Metaspace keeps whole grapheme clusters together (it never splits inside a cluster,
    # unlike ByteLevel which cuts multi-byte Bengali codepoints apart).
    tok.pre_tokenizer = pre_tokenizers.Sequence([
        pre_tokenizers.Punctuation(behavior="isolated"),
        pre_tokenizers.Metaspace(replacement="▁", prepend_scheme="always"),
    ])
    tok.decoder = decoders.Metaspace(replacement="▁", prepend_scheme="always")
    tok.train_from_iterator((t for t in texts), trainer=trainer, length=len(texts))
    if cfg.grapheme_safe_vocab and kind == "unigram":
        try:
            tok = enforce_grapheme_safe_vocab(tok, specials, vocab_size)
        except Exception as e:      # tokenizers JSON schema differs across versions
            log(f"WARNING: grapheme-safe pruning skipped ({type(e).__name__}: {e}). "
                "Vocabulary is unconstrained — do NOT claim cluster safety in the paper.")
    tok.post_processor = processors.TemplateProcessing(
        single="<s> $A </s>", pair="<s> $A </s> </s> $B </s>",
        special_tokens=[("<s>", tok.token_to_id("<s>")), ("</s>", tok.token_to_id("</s>"))],
    )
    fast = PreTrainedTokenizerFast(
        tokenizer_object=tok, unk_token="<unk>", pad_token="<pad>",
        cls_token="<s>", sep_token="</s>", mask_token="<mask>", bos_token="<s>", eos_token="</s>",
        model_max_length=cfg.max_len,
    )
    return fast

with stage("tokenizer"):
    if not stage_done("tokenizer"):
        pre_texts = df.loc[df.split == "pretrain", "text"].tolist()
        log(f"training {cfg.tokenizer_model} tokenizer on {len(pre_texts):,} pretraining chunks")
        nctb_tok = train_tokenizer(pre_texts, cfg.vocab_size, cfg.tokenizer_model)
        TOK_DIR.mkdir(parents=True, exist_ok=True)
        nctb_tok.save_pretrained(TOK_DIR)

nctb_tok = AutoTokenizer.from_pretrained(TOK_DIR)
log(f"NCTB tokenizer: |V| = {nctb_tok.vocab_size:,}")
_demo = df.loc[df.language == "bn", "text"].iloc[0][:160]
print("sample :", _demo)
print("tokens :", nctb_tok.tokenize(_demo)[:28])

# ---- verify the cluster-safety claim instead of asserting it in prose --------
def cluster_split_rate(tokenizer, texts) -> float:
    """Fraction of emitted tokens that begin with a combining mark, i.e. that
    strand a matra/hasant from its base. This is the number the paper should
    report; it must be ~0 for the grapheme-safe claim to hold."""
    bad = tot = 0
    for t in texts:
        for p in tokenizer.tokenize(t):
            # strip every continuation marker so WordPiece ("##ি"), SentencePiece
            # ("▁") and raw pieces are judged on the same footing -- without this
            # a WordPiece tokenizer scores a spurious 0.0
            body = p[2:] if p.startswith("##") else (p[1:] if p.startswith("▁") else p)
            tot += 1
            if body and _MARK_INITIAL.match(body): bad += 1
    return bad / max(tot, 1)

_bn = df.loc[df.language == "bn", "text"].head(400).tolist()
print(f"\ncluster-split rate (ours): {cluster_split_rate(nctb_tok, _bn):.4f}  "
      "(fraction of tokens starting with a combining mark; must be ~0)")
print("baseline comparison is in the next cell")


In [ ]:
# ============================================================================
# CELL 12 — Fertility / compression benchmark  (Figure 3)
# ============================================================================
def hf_tok(name):
    kw = dict(use_fast=True)
    if cfg.model_cache: kw["cache_dir"] = cfg.model_cache
    return AutoTokenizer.from_pretrained(name, **kw)

def fertility_stats(tokenizer, texts) -> Dict[str, float]:
    n_words = n_toks = n_chars = 0
    n_unk = 0
    unk_id = getattr(tokenizer, "unk_token_id", None)
    for t in texts:
        ids = tokenizer(t, add_special_tokens=False)["input_ids"]
        n_toks += len(ids); n_words += len(t.split()); n_chars += len(t)
        if unk_id is not None: n_unk += sum(1 for i in ids if i == unk_id)
    return {"fertility": n_toks / max(n_words, 1),
            "chars_per_token": n_chars / max(n_toks, 1),
            "unk_rate": n_unk / max(n_toks, 1),
            "cluster_split": cluster_split_rate(tokenizer, texts[:400]),
            "vocab_size": int(getattr(tokenizer, "vocab_size", len(tokenizer)))}

with stage("fertility"):
    if not stage_done("fertility"):
        rng = np.random.default_rng(cfg.seed)
        sample = {}
        for lang in ["bn", "en"]:
            pool = df.loc[(df.language == lang) & (df.split != "pretrain"), "text"].tolist()
            if pool: sample[lang] = list(rng.choice(pool, size=min(1500, len(pool)), replace=False))
        rows = []
        cands = [("NCTB-BERT (ours)", nctb_tok)]
        for b in cfg.baselines:
            try: cands.append((b, hf_tok(b)))
            except Exception as e: log(f"tokenizer unavailable: {b} ({e})")
        for name, tk in cands:
            for lang, texts in sample.items():
                r = fertility_stats(tk, texts); r.update(model=name, language=lang); rows.append(r)
        fert = pd.DataFrame(rows)
        save_art(fert, "fertility.csv")

fert = load_art("fertility.csv")
display(fert.pivot(index="model", columns="language", values="fertility").round(3))
print("\ncluster-split rate by tokenizer (Bengali; lower is better):")
display(fert[fert.language == "bn"].set_index("model")[["cluster_split", "unk_rate"]].round(4))

# ---- Figure 3 -------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.4))
piv = fert.pivot(index="model", columns="language", values="fertility")
piv = piv.sort_values(piv.columns[0])
y = np.arange(len(piv)); w = 0.38
ax = axes[0]
for i, lang in enumerate([c for c in ["bn", "en"] if c in piv.columns]):
    vals = piv[lang].values
    hl = [PAL[i] if m.startswith("NCTB") else (SEQ[2] if i == 0 else "#f0b79a") for m in piv.index]
    ax.barh(y + (i - 0.5) * (w + 0.02), vals, w, color=hl,
            label={"bn": "Bengali", "en": "English"}[lang])
    for yi, v in zip(y + (i - 0.5) * (w + 0.02), vals):
        ax.text(v + 0.02, yi, f"{v:.2f}", va="center", fontsize=7.3, color=INK2)
ax.set_yticks(y); ax.set_yticklabels([m.split("/")[-1] for m in piv.index], fontsize=8)
ax.legend(loc="lower right")
finish(ax, "Tokenizer fertility (lower is better)", "subword tokens per whitespace word",
       "fertility", None, ygrid=False, xgrid=True)

ax = axes[1]
vs = fert.groupby("model")["vocab_size"].first().reindex(piv.index) / 1000
cols = [PAL[0] if m.startswith("NCTB") else MUTED for m in piv.index]
ax.barh(y, vs.values, 0.6, color=cols)
for yi, v in zip(y, vs.values):
    ax.text(v + 1, yi, f"{v:,.0f}k", va="center", fontsize=7.5, color=INK2)
ax.set_yticks(y); ax.set_yticklabels([])
finish(ax, "Vocabulary size", "thousand tokens", "|V| (k)", None, ygrid=False, xgrid=True)
fig.tight_layout(); savefig(fig, "fig3_tokenizer_fertility"); plt.show()


---
## Section 5 — FOCUS-style embedding transfer (contribution C3, part 2)

Swapping the vocabulary normally destroys the pretrained embedding matrix. We initialise the new
32k matrix following **FOCUS** (Dobler & de Melo, EMNLP 2023):

1. Build an **auxiliary embedding space** $a_t$ for every new token $t$, from the pretraining pool
   tokenised with the new NCTB tokenizer. FOCUS uses fastText here; we use an equivalent but
   dependency-free and fully deterministic construction — truncated SVD of the PPMI co-occurrence
   matrix (which is what skip-gram implicitly factorises, Levy & Goldberg 2014) concatenated with
   truncated SVD of character 2–5-gram TF-IDF (fastText's sub-character signal, which matters for
   Bengali conjuncts and keeps rare tokens from receiving a zero vector).
2. **Overlapping tokens** ($t \in V_{\text{new}} \cap V_{\text{src}}$) copy their pretrained XLM-R vector
   directly — these become *anchors*.
3. **Non-overlapping tokens** are a sparse convex combination of anchors, weighted by auxiliary-space
   cosine similarity with a top-$k$ sparsemax-like renormalisation:

$$
e_t \;=\; \sum_{s\in \mathcal N_k(t)} \alpha_{ts}\, e_s ,\qquad
\alpha_{ts} = \frac{\exp\big(\cos(a_t,a_s)/\tau\big)}{\sum_{s'\in\mathcal N_k(t)}\exp\big(\cos(a_t,a_{s'})/\tau\big)}
$$

All non-embedding weights (encoder blocks) are copied unchanged from XLM-R, so the model starts from a
strong multilingual initialisation with a 8× smaller embedding table.


In [ ]:
# ============================================================================
# CELL 13 — FOCUS embedding transfer -> NCTB-BERT initial checkpoint
# ============================================================================


class AuxSpace:
    """
    Auxiliary embedding space for FOCUS, built with numpy/scipy/sklearn only
    (no fastText / gensim dependency, which is the most fragile package on
    Kaggle images and pins old numpy).

    Two complementary halves, L2-normalised then concatenated:
      * distributional  — PPMI over a symmetric ±w co-occurrence matrix of the
        NEW tokenizer's ids on the pretraining pool, reduced by truncated SVD.
        This is what fastText's skip-gram objective approximates (Levy &
        Goldberg, 2014), so we get the same signal deterministically.
      * sub-character   — TF-IDF over character 2-5 grams of the token surface
        form, reduced by truncated SVD. This is what fastText's character n-gram
        buckets provide, and it keeps rare / unseen tokens from having a zero
        vector — essential for Bengali conjuncts.
    """
    def __init__(self, vectors: np.ndarray, id2tok: Dict[int, str]):
        self.V = vectors; self.id2tok = id2tok
        self.tok2id = {t: i for i, t in id2tok.items()}
        self.has = np.linalg.norm(vectors, axis=1) > 1e-8
    def __contains__(self, tok): return self.tok2id.get(tok) is not None and self.has[self.tok2id[tok]]
    def __getitem__(self, tok): return self.V[self.tok2id[tok]]

def build_aux_space(texts, tokenizer, dim_dist=200, dim_char=100,
                    window=5, seed=0, batch=2000) -> AuxSpace:
    V = len(tokenizer.get_vocab())
    id2tok = {i: t for t, i in tokenizer.get_vocab().items()}

    # ---- 1. co-occurrence counts (batched, sparse) -------------------------
    M = sps.csr_matrix((V, V), dtype=np.float32)
    rows, cols = [], []
    def flush():
        nonlocal M, rows, cols
        if not rows: return
        rr = np.concatenate(rows); cc = np.concatenate(cols)
        M = M + sps.coo_matrix((np.ones(len(rr), dtype=np.float32), (rr, cc)),
                               shape=(V, V)).tocsr()
        rows, cols = [], []
    for n, t in enumerate(tqdm(texts, desc="co-occurrence")):
        ids = np.asarray(tokenizer(t, add_special_tokens=False)["input_ids"], dtype=np.int64)
        if ids.size < 2: continue
        for off in range(1, window + 1):
            if ids.size <= off: break
            a_, b_ = ids[:-off], ids[off:]
            rows.append(np.concatenate([a_, b_])); cols.append(np.concatenate([b_, a_]))
        if (n + 1) % batch == 0: flush()
    flush()

    # ---- 2. PPMI + truncated SVD ------------------------------------------
    M = M.tocoo()
    tot = max(float(M.data.sum()), 1.0)
    r_sum = np.asarray(M.tocsr().sum(axis=1)).ravel() + 1e-9
    c_sum = np.asarray(M.tocsr().sum(axis=0)).ravel() + 1e-9
    ppmi_data = np.log(np.maximum(M.data * tot / (r_sum[M.row] * c_sum[M.col]), 1e-12))
    ppmi_data = np.maximum(ppmi_data, 0.0)
    keep = ppmi_data > 0
    PPMI = sps.csr_matrix((ppmi_data[keep], (M.row[keep], M.col[keep])), shape=(V, V))
    d1 = min(dim_dist, max(2, min(PPMI.shape) - 1))
    dist = TruncatedSVD(n_components=d1, random_state=seed).fit_transform(PPMI).astype(np.float32)

    # ---- 3. character n-gram half -----------------------------------------
    surfaces = [id2tok.get(i, "").replace("▁", " ") for i in range(V)]
    tv = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=1, max_features=60_000)
    Xc = tv.fit_transform(surfaces)
    d2 = min(dim_char, max(2, min(Xc.shape) - 1))
    char = TruncatedSVD(n_components=d2, random_state=seed).fit_transform(Xc).astype(np.float32)

    def _n(x):
        return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-9)
    aux = np.hstack([_n(dist), _n(char)]).astype(np.float32)
    return AuxSpace(aux, id2tok)

def focus_init(src_model_name, new_tok, aux, topk, temp) -> Tuple[Any, dict]:
    kw = {"cache_dir": cfg.model_cache} if cfg.model_cache else {}
    src_tok = AutoTokenizer.from_pretrained(src_model_name, **kw)
    src = AutoModelForMaskedLM.from_pretrained(src_model_name, **kw)
    src_emb = src.get_input_embeddings().weight.detach().float().cpu().numpy()

    src_vocab = src_tok.get_vocab()
    new_vocab = new_tok.get_vocab()
    dim = src_emb.shape[1]
    new_emb = np.zeros((len(new_vocab), dim), dtype=np.float32)

    # map special tokens explicitly
    special_map = {"<pad>": src_tok.pad_token, "<unk>": src_tok.unk_token, "<s>": src_tok.cls_token,
                   "</s>": src_tok.sep_token, "<mask>": src_tok.mask_token}
    overlap, missing = {}, []
    for tok, idx in new_vocab.items():
        cand = special_map.get(tok, tok)
        # XLM-R is SentencePiece and also uses U+2581 as the word-start marker,
        # so surface forms line up directly; fall back to the bare form otherwise.
        if cand in src_vocab:            overlap[idx] = src_vocab[cand]
        elif cand.lstrip("▁") in src_vocab: overlap[idx] = src_vocab[cand.lstrip("▁")]
        else:                            missing.append(idx)

    for n_idx, s_idx in overlap.items():
        new_emb[n_idx] = src_emb[s_idx]

    # anchors in auxiliary space
    inv_new = {v: k for k, v in new_vocab.items()}
    anchor_ids = [i for i in overlap if inv_new[i] in aux]
    if len(anchor_ids) < 32:
        log("WARNING: too few anchors for FOCUS — falling back to subword-average init")
        anchor_ids = []
    if anchor_ids:
        A = np.stack([aux[inv_new[i]] for i in anchor_ids]).astype(np.float32)
        A /= (np.linalg.norm(A, axis=1, keepdims=True) + 1e-9)
        E = np.stack([new_emb[i] for i in anchor_ids])
        for i in tqdm(missing, desc="FOCUS init"):
            tok = inv_new[i]
            if tok in aux:
                q = aux[tok].astype(np.float32); q /= (np.linalg.norm(q) + 1e-9)
                sims = A @ q
                k = min(topk, len(sims))
                nb = np.argpartition(-sims, k - 1)[:k]
                wts = np.exp(sims[nb] / temp); wts /= wts.sum()
                new_emb[i] = wts @ E[nb]
            else:  # unseen token: average of its source-tokenizer pieces
                pieces = src_tok(tok.replace("▁", " "), add_special_tokens=False)["input_ids"]
                new_emb[i] = src_emb[pieces].mean(0) if pieces else src_emb.mean(0)
    else:
        for i in missing:
            pieces = src_tok(inv_new[i].replace("▁", " "), add_special_tokens=False)["input_ids"]
            new_emb[i] = src_emb[pieces].mean(0) if pieces else src_emb.mean(0)

    # rebuild the model with the new vocabulary
    conf = AutoConfig.from_pretrained(src_model_name, **kw)
    conf.vocab_size = len(new_vocab)
    conf.pad_token_id = new_vocab["<pad>"]; conf.bos_token_id = new_vocab["<s>"]
    conf.eos_token_id = new_vocab["</s>"]; conf.max_position_embeddings = max(
        conf.max_position_embeddings, cfg.max_len + 8)
    model = AutoModelForMaskedLM.from_config(conf)
    # copy every non-embedding weight from the source
    sd_src, sd_new = src.state_dict(), model.state_dict()
    copied = 0
    for k, v in sd_src.items():
        if k in sd_new and sd_new[k].shape == v.shape:
            sd_new[k] = v.clone(); copied += 1
    model.load_state_dict(sd_new)
    with torch.no_grad():
        model.get_input_embeddings().weight.copy_(torch.tensor(new_emb))
        if model.get_output_embeddings() is not None and \
           model.get_output_embeddings().weight.shape == model.get_input_embeddings().weight.shape:
            model.tie_weights()
    stats = {"new_vocab": len(new_vocab), "overlap": len(overlap), "focus_init": len(missing),
             "tensors_copied": copied,
             "src_embed_params": int(src_emb.size), "new_embed_params": int(new_emb.size),
             "embed_param_reduction": float(1 - new_emb.size / src_emb.size)}
    del src; gc.collect()
    return model, stats

# ---------------------------------------------------------------------------
# build_init(): materialise a starting checkpoint for DAPT.
#
#   vocab_transfer=False -> the base model with its OWN tokenizer (default).
#   vocab_transfer=True  -> the NCTB 32k vocabulary, FOCUS-initialised.
#
# Native vocabulary is the default for the main model deliberately. Swapping the
# vocabulary AND reordering the data in the same run would confound the
# curriculum claim with a representation change; the vocabulary swap is measured
# separately as its own arm.
# ---------------------------------------------------------------------------
_AUX_CACHE = {}

def get_aux_space():
    if "aux" not in _AUX_CACHE:
        pre_texts = df.loc[df.split == "pretrain", "text"].tolist()
        if cfg.smoke_test: pre_texts = pre_texts[:4000]
        _AUX_CACHE["aux"] = build_aux_space(
            pre_texts, nctb_tok,
            dim_dist=int(cfg.focus_aux_dim * 2 / 3),
            dim_char=int(cfg.focus_aux_dim * 1 / 3), seed=cfg.seed)
    return _AUX_CACHE["aux"]

def build_init(base_model: str, vocab_transfer: bool) -> Path:
    tag = base_model.split("/")[-1] + ("_focus" if vocab_transfer else "_native")
    out = OUT / "models" / f"init_{tag}"
    if (out / "config.json").exists(): return out
    kw = {"cache_dir": cfg.model_cache} if cfg.model_cache else {}
    if vocab_transfer:
        model, st = focus_init(base_model, nctb_tok, get_aux_space(),
                               cfg.focus_topk, cfg.focus_temperature)
        out.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(out); nctb_tok.save_pretrained(out)
        save_art(st, f"focus_stats_{tag}.json")
        log(f"[{tag}] FOCUS: {st['overlap']:,} anchors, {st['focus_init']:,} interpolated, "
            f"embedding params -{st['embed_param_reduction']*100:.1f}%")
        del model
    else:
        model = AutoModelForMaskedLM.from_pretrained(base_model, **kw)
        tk = AutoTokenizer.from_pretrained(base_model, **kw)
        out.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(out); tk.save_pretrained(out)
        log(f"[{tag}] native checkpoint materialised (|V|={tk.vocab_size:,})")
        del model
    gc.collect(); torch.cuda.empty_cache()
    return out

log("build_init() ready — checkpoints are materialised on demand by later cells")


---
## Section 6 — GRADE-CL pretraining (the core contribution, C1)

### Competence-based pacing

Following the competence framework of Platanios et al. (2019), at optimisation step $t$ the model is
allowed to sample from the *easiest* $c(t)$ proportion of the corpus under $D(x)$:

$$
c(t)=\min\!\left(1,\;\Big(\tfrac{t}{T_c}\big(1-c_0^{\,p}\big)+c_0^{\,p}\Big)^{1/p}\right),
\qquad c_0=0.15,\; p=2,\; T_c = 0.75\,T .
$$

### Two-phase vocabulary transfer

FOCUS replaces ~97% of the embedding parameters. Back-propagating into the whole network from step 0
destroys that initialisation before the encoder can adapt to it — empirically the model's held-out
pseudo-perplexity *rose* above its own starting point. Training therefore runs in two phases:

1. **Embeddings only** (first $15\%$ of steps): the encoder body is frozen and the embedding matrix
   is trained at a higher LR ($5\times10^{-4}$), letting the new vocabulary align to the frozen
   representation space.
2. **Full model** (remaining steps): everything unfreezes and the embedding LR drops to the base rate.

The step budget is derived from a target number of *epochs* over the pretraining pool rather than
fixed, with early stopping on held-out MLM loss — a 28k-chunk pool cannot absorb a fixed 20k-step
budget without severe overfitting.

### Anti-forgetting replay

A pure easy→hard schedule catastrophically forgets early-grade material. Each batch is therefore
mixed: a fraction $(1-\rho)$ is drawn from the **frontier** (the newly unlocked difficulty band,
$D\in[c(t)-\delta,\,c(t)]$) and $\rho=0.25$ from **mastered** material ($D<c(t)-\delta$). This
"spiral curriculum" mirrors how the NCTB curriculum itself revisits concepts across grades — the
pedagogical analogy is not decorative, it is the reason the term is there.

### Arms

| arm | ordering signal |
|---|---|
| `grade_cl` | **ours** — $D(x)$, grade-anchored, with replay |
| `random` | i.i.d. shuffling (standard DAPT) |
| `anti` | $1-D(x)$ (hard→easy) — tests whether *any* ordering helps or only the right one |
| `length_cl` | chunk length percentile (the classic proxy curriculum) |
| `no_dapt` | no pretraining; FOCUS-initialised weights used directly |


In [ ]:
# ============================================================================
# CELL 14 — Pretraining dataset, whole-word masking collator, optional objectives
# ============================================================================
def pack_chunks(frame: pd.DataFrame, tokenizer, max_len: int, diff_col: str) -> pd.DataFrame:
    """
    Concatenate consecutive chunks WITHIN a (book, chapter) up to `max_len`
    tokens, never crossing a document boundary.

    Each chunk is short, so one-chunk-per-example wastes most of the 256-token
    window on padding and denies the model any cross-chunk context. Packing
    within a chapter recovers both without the usual cost of packing (mixing
    unrelated documents into one attention window). Difficulty and OCR agreement
    are length-weighted means of the constituents, so the curriculum signal
    survives the merge. BanglaBERT applies the same document-boundary rule.
    """
    rows = []
    budget = max_len - 8
    for (bk, ch), g in frame.sort_values(["book_id", "chapter_no", "chunk_index"]).groupby(
            ["book_id", "chapter_no"], sort=False):
        buf, blen = [], 0
        def flush():
            if not buf: return
            w = np.array([len(x["text"]) for x in buf], dtype=float); w /= w.sum()
            rows.append({
                "text": " ".join(x["text"] for x in buf),
                "book_id": bk, "chapter_no": ch, "language": buf[0]["language"],
                "grade": float(np.average([x["grade"] for x in buf], weights=w)),
                diff_col: float(np.average([x[diff_col] for x in buf], weights=w)),
                "ocr_agreement": float(np.average(
                    [x["ocr_agreement"] if x["ocr_agreement"] == x["ocr_agreement"] else 1.0
                     for x in buf], weights=w)),
                "n_packed": len(buf)})
        for _, r in g.iterrows():
            n = len(tokenizer(r["text"], add_special_tokens=False)["input_ids"])
            if blen + n > budget and buf:
                flush(); buf, blen = [], 0
            buf.append(r); blen += n
        flush()
    return pd.DataFrame(rows)

class NCTBPretrainDataset(TorchDataset):
    def __init__(self, frame: pd.DataFrame, tokenizer, max_len: int, diff_col: str):
        self.tok = tokenizer; self.max_len = max_len
        self.texts = frame["text"].tolist()
        self.diff  = frame[diff_col].to_numpy(dtype=np.float32)
        self.agree = frame["ocr_agreement"].fillna(1.0).to_numpy(dtype=np.float32)
        self.grade = frame["grade"].to_numpy(dtype=np.float32)
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(self.texts[i], truncation=True, max_length=self.max_len,
                       padding=False, return_special_tokens_mask=True)
        enc = {k: v for k, v in enc.items()}
        enc["ocr_weight"] = float(self.agree[i])
        enc["difficulty"] = float(self.diff[i])
        return enc

class WholeWordMLMCollator:
    """Dynamic whole-word masking (Metaspace '▁' marks word starts) + optional OCR loss weights."""
    def __init__(self, tokenizer, mlm_prob, whole_word=True, ocr_weighted=False, floor=0.5):
        self.tok = tokenizer; self.p = mlm_prob; self.ww = whole_word
        self.ocr_weighted = ocr_weighted; self.floor = floor
        self.mask_id = tokenizer.mask_token_id
        self.vocab = tokenizer.vocab_size
        inv = {v: k for k, v in tokenizer.get_vocab().items()}
        # Word-start detection must work for BOTH tokenizer families, because the
        # base model is now chosen empirically: SentencePiece marks word STARTS
        # with "▁" (XLM-R, MuRIL-SP), WordPiece marks CONTINUATIONS with "##"
        # (mBERT, bangla-bert-base). Assuming only "▁" silently disabled
        # whole-word masking on every WordPiece model.
        pieces = [inv.get(i, "") for i in range(len(inv))]
        sp = sum(1 for p in pieces if p.startswith("▁"))
        wp = sum(1 for p in pieces if p.startswith("##"))
        self.scheme = "sentencepiece" if sp >= wp else "wordpiece"
        if self.scheme == "sentencepiece":
            starts = [1 if p.startswith("▁") else 0 for p in pieces]
        else:
            starts = [0 if p.startswith("##") else 1 for p in pieces]
        self.is_word_start = np.array(starts, dtype=np.int8)
        self.special_ids = set(tokenizer.all_special_ids)

    def _word_spans(self, ids):
        spans, cur = [], []
        for pos, tid in enumerate(ids):
            if tid in self.special_ids: continue
            if self.is_word_start[tid] and cur: spans.append(cur); cur = [pos]
            else: cur.append(pos)
        if cur: spans.append(cur)
        return spans

    def __call__(self, features):
        ocr_w = torch.tensor([f.pop("ocr_weight", 1.0) for f in features], dtype=torch.float)
        for f in features: f.pop("difficulty", None)
        batch = self.tok.pad(features, return_tensors="pt", pad_to_multiple_of=8)
        special = batch.pop("special_tokens_mask")
        labels = batch["input_ids"].clone()
        B, L = labels.shape
        mask = torch.zeros(B, L, dtype=torch.bool)
        for b in range(B):
            ids = batch["input_ids"][b].tolist()
            units = self._word_spans(ids) if self.ww else \
                    [[i] for i, t in enumerate(ids) if t not in self.special_ids]
            if not units: continue
            n_target = max(1, int(round(self.p * sum(len(u) for u in units))))
            order = torch.randperm(len(units)).tolist()
            picked = 0
            for u in order:
                if picked >= n_target: break
                mask[b, units[u]] = True; picked += len(units[u])
        mask &= ~special.bool()
        labels[~mask] = -100
        # 80/10/10
        r = torch.rand(B, L)
        batch["input_ids"][mask & (r < 0.8)] = self.mask_id
        rnd = (r >= 0.8) & (r < 0.9) & mask
        batch["input_ids"][rnd] = torch.randint(0, self.vocab, (int(rnd.sum()),))
        batch["labels"] = labels
        if self.ocr_weighted:
            batch["ocr_weight"] = self.floor + (1 - self.floor) * ocr_w.clamp(0, 1)
        return batch

log("collator ready")


In [ ]:
# ============================================================================
# CELL 15 — GRADE-CL competence sampler (contribution C1)
# ============================================================================
class CompetenceCurriculumSampler(Sampler):
    """
    Yields indices for `total_batches` batches. At step t the competence c(t) defines
    the unlocked difficulty quantile. Each batch mixes:
        (1-rho) frontier samples : D in [c(t)-delta, c(t)]
        rho     mastered samples : D <  c(t)-delta   (spiral-curriculum replay)
    mode: 'grade_cl' | 'anti' | 'length_cl' | 'random'
    """
    def __init__(self, difficulty: np.ndarray, batch_size: int, total_batches: int,
                 mode: str, c0: float, power: float, curriculum_frac: float,
                 replay_ratio: float, n_bins: int, seed: int):
        self.d = np.asarray(difficulty, dtype=np.float64)
        if mode == "anti": self.d = 1.0 - self.d
        self.mode = mode
        self.bs = batch_size; self.tb = total_batches
        self.c0, self.p = c0, power
        self.Tc = max(1, int(curriculum_frac * total_batches))
        self.rho = replay_ratio
        self.rng = np.random.default_rng(seed)
        self.order = np.argsort(self.d, kind="stable")     # easiest -> hardest
        self.N = len(self.d)
        self.delta_n = max(self.bs, self.N // n_bins)

    def competence(self, t: int) -> float:
        if self.mode == "random": return 1.0
        return min(1.0, ((t / self.Tc) * (1 - self.c0 ** self.p) + self.c0 ** self.p) ** (1 / self.p))

    def __iter__(self):
        for t in range(self.tb):
            c = self.competence(t)
            hi = max(self.bs, int(c * self.N))
            lo = max(0, hi - self.delta_n)
            n_replay = int(round(self.rho * self.bs)) if (self.mode != "random" and lo > 0) else 0
            n_front = self.bs - n_replay
            front = self.rng.choice(self.order[lo:hi], size=n_front,
                                    replace=n_front > (hi - lo))
            idx = [front]
            if n_replay:
                idx.append(self.rng.choice(self.order[:lo], size=n_replay, replace=n_replay > lo))
            batch = np.concatenate(idx); self.rng.shuffle(batch)
            for i in batch: yield int(i)

    def __len__(self): return self.tb * self.bs

# --- sanity/diagnostic: what the schedule actually looks like ---------------
_sched = CompetenceCurriculumSampler(df["difficulty"].values, 32, 2000, "grade_cl",
                                     cfg.c0, cfg.competence_power, cfg.curriculum_frac,
                                     cfg.replay_ratio, cfg.difficulty_bins, cfg.seed)
_steps = np.arange(0, 2000, 10)
_comp  = [_sched.competence(t) for t in _steps]
fig, ax = plt.subplots(figsize=(5.2, 3.0))
ax.plot(_steps / 2000, _comp, color=PAL[0])
ax.fill_between(_steps / 2000, 0, _comp, color=PAL[0], alpha=0.12, lw=0)
ax.axhline(1.0, color=MUTED, lw=0.8, ls=":")
ax.text(0.02, cfg.c0 + 0.03, f"c₀ = {cfg.c0}", fontsize=8, color=INK2)
finish(ax, "GRADE-CL competence schedule", f"p={cfg.competence_power}, ramp over {cfg.curriculum_frac:.0%} of training",
       "training progress", "unlocked difficulty quantile c(t)")
fig.tight_layout(); savefig(fig, "fig4_competence_schedule"); plt.show()


In [ ]:
# ============================================================================
# CELL 16 — Pretraining loop (fp16, multi-GPU, budget-aware, resumable)
# ============================================================================
def masked_mlm_loss(logits, labels, weights=None):
    V = logits.size(-1)
    ll = F.cross_entropy(logits.view(-1, V), labels.view(-1), ignore_index=-100, reduction="none")
    ll = ll.view(labels.shape)
    valid = (labels != -100).float()
    if weights is None:
        return ll.sum() / valid.sum().clamp(min=1)
    w = weights.unsqueeze(1).expand_as(ll)
    return (ll * valid * w).sum() / (valid * w).sum().clamp(min=1)

@torch.no_grad()
def eval_mlm(model, loader, tok, max_batches=40):
    """
    Held-out MLM loss under PURE masking.

    The training collator uses the standard 80/10/10 rule, so ~10% of scored
    positions still carry their original token and are trivially predictable.
    That inflates the apparent quality of a weak model and made the in-training
    number (ppl ~12) irreconcilable with strided pseudo-perplexity (ppl ~700).
    Here every scored position is replaced by [MASK], which makes this metric
    directly comparable with `pseudo_perplexity` later in the notebook.
    """
    model.eval(); tot, n = 0.0, 0
    for i, b in enumerate(loader):
        if i >= max_batches: break
        b = {k: v.to(DEVICE) for k, v in b.items() if k != "ocr_weight"}
        scored = b["labels"] != -100
        b["input_ids"] = b["input_ids"].masked_fill(scored, tok.mask_token_id)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
            out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"])
            loss = masked_mlm_loss(out.logits.float(), b["labels"])
        tot += loss.item(); n += 1
    model.train()
    return tot / max(n, 1)

def set_body_frozen(model, frozen: bool):
    """Phase 1 of vocabulary transfer: train embeddings only."""
    core = model.module if isinstance(model, nn.DataParallel) else model
    emb = core.get_input_embeddings()
    emb_ids = {id(p) for p in emb.parameters()}
    out_emb = core.get_output_embeddings()
    if out_emb is not None: emb_ids |= {id(p) for p in out_emb.parameters()}
    for p in core.parameters():
        p.requires_grad = True if id(p) in emb_ids else (not frozen)

def pretrain_arm(arm: str, init_dir: Optional[Path] = None,
                 steps_override: Optional[int] = None,
                 out_name: Optional[str] = None,
                 mode_override: Optional[str] = None) -> Path:
    init_dir = Path(init_dir) if init_dir is not None else INIT_DIR
    arm_dir = OUT / "models" / (out_name or f"nctb_bert_{arm}")
    ck_dir  = arm_dir / "ckpt"
    if arm == "no_dapt":
        if not arm_dir.exists():
            shutil.copytree(init_dir, arm_dir)
        return arm_dir
    arm_dir.mkdir(parents=True, exist_ok=True); ck_dir.mkdir(exist_ok=True)

    set_all_seeds(cfg.seed)
    tok = AutoTokenizer.from_pretrained(init_dir)
    model = AutoModelForMaskedLM.from_pretrained(init_dir).to(DEVICE)
    if N_GPU > 1: model = nn.DataParallel(model)

    diff_col = "difficulty_len" if arm == "length_cl" else "difficulty"
    tr = df[df.split == "pretrain"].reset_index(drop=True)
    ev = df[df.split == "ds_val"].reset_index(drop=True)
    if cfg.pack_sequences:
        n0 = len(tr)
        tr = pack_chunks(tr, tok, cfg.max_len, diff_col)
        ev = pack_chunks(ev, tok, cfg.max_len, diff_col)
        log(f"[{arm}] packed {n0:,} chunks -> {len(tr):,} sequences "
            f"({n0/max(len(tr),1):.1f} chunks/sequence)")
    ds_tr = NCTBPretrainDataset(tr, tok, cfg.max_len, diff_col)
    ds_ev = NCTBPretrainDataset(ev, tok, cfg.max_len, diff_col)
    coll = WholeWordMLMCollator(tok, cfg.mlm_prob, cfg.whole_word_mask,
                                cfg.use_ocr_weighted_mlm, cfg.ocr_weight_floor)

    bs = cfg.per_device_bs * max(N_GPU, 1)
    # ---- derive the step budget from EPOCHS over the pool, not a fixed number.
    # A hard 20k steps was ~180 epochs over a 28k-chunk pool: guaranteed overfit.
    per_step = bs * cfg.grad_accum
    total_steps = int(math.ceil(cfg.target_epochs * len(ds_tr) / per_step))
    total_steps = int(min(max(total_steps, cfg.min_steps), cfg.max_steps_cap))
    if steps_override: total_steps = int(steps_override)
    warm_steps  = int(cfg.embed_warmup_frac * total_steps)
    log(f"[{arm}] pool={len(ds_tr):,} chunks · {per_step} ex/step · "
        f"{total_steps} steps ≈ {total_steps*per_step/len(ds_tr):.1f} epochs "
        f"(first {warm_steps} = embeddings-only warmup)")
    total_batches = total_steps * cfg.grad_accum
    sampler = CompetenceCurriculumSampler(
        ds_tr.diff, bs, total_batches,
        mode=(mode_override or ("random" if arm in ("random", "vocab_swap") else arm)),
        c0=cfg.c0, power=cfg.competence_power, curriculum_frac=cfg.curriculum_frac,
        replay_ratio=(0.0 if (mode_override or arm) == "random" else cfg.replay_ratio),
        n_bins=cfg.difficulty_bins, seed=cfg.seed)
    dl_tr = DataLoader(ds_tr, batch_size=bs, sampler=sampler, collate_fn=coll,
                       num_workers=2, pin_memory=True, drop_last=True)
    dl_ev = DataLoader(ds_ev, batch_size=bs, shuffle=False, collate_fn=coll, num_workers=2)

    # Two parameter groups: embeddings (trained throughout, high LR during the
    # warmup phase) and the encoder body (frozen for the first `warm_steps`).
    core0 = model.module if isinstance(model, nn.DataParallel) else model
    emb_ids = {id(p) for p in core0.get_input_embeddings().parameters()}
    if core0.get_output_embeddings() is not None:
        emb_ids |= {id(p) for p in core0.get_output_embeddings().parameters()}
    emb_params, decay, nodecay = [], [], []
    for n_, p in core0.named_parameters():
        if id(p) in emb_ids: emb_params.append(p)
        elif any(k in n_ for k in ["bias", "LayerNorm.weight", "layer_norm"]): nodecay.append(p)
        else: decay.append(p)
    opt = torch.optim.AdamW(
        [{"params": emb_params, "weight_decay": 0.0,               "lr": cfg.embed_warmup_lr},
         {"params": decay,      "weight_decay": cfg.weight_decay,  "lr": cfg.lr},
         {"params": nodecay,    "weight_decay": 0.0,               "lr": cfg.lr}], eps=1e-6)
    sch = get_linear_schedule_with_warmup(opt, int(cfg.warmup_ratio * total_steps), total_steps)
    scaler = make_scaler(torch.cuda.is_available())
    set_body_frozen(model, True)          # phase 1 begins

    start_step = 0
    ck = ck_dir / "state.pt"
    if ck.exists():
        st = torch.load(ck, map_location="cpu")
        (model.module if isinstance(model, nn.DataParallel) else model).load_state_dict(st["model"])
        opt.load_state_dict(st["opt"]); sch.load_state_dict(st["sch"]); scaler.load_state_dict(st["scaler"])
        start_step = st["step"]
        log(f"[{arm}] resumed from step {start_step}")

    history = json.loads((arm_dir / "history.json").read_text()) if (arm_dir / "history.json").exists() else []
    best_loss, best_state, since_improve = float("inf"), None, 0
    if start_step >= warm_steps:
        set_body_frozen(model, False)
        for g in opt.param_groups[:1]: g["lr"] = cfg.lr
    model.train()
    it = iter(dl_tr)
    # fast-forward the sampler when resuming
    for _ in range(start_step * cfg.grad_accum):
        try: next(it)
        except StopIteration: it = iter(dl_tr); break

    pbar = tqdm(range(start_step, total_steps), desc=f"pretrain[{arm}]", initial=start_step,
                total=total_steps)
    running = []
    for step in pbar:
        if step == warm_steps:            # phase 2: unfreeze, drop embedding LR
            set_body_frozen(model, False)
            opt.param_groups[0]["lr"] = cfg.lr
            log(f"[{arm}] step {step}: encoder unfrozen, embedding LR -> {cfg.lr}")
        opt.zero_grad(set_to_none=True)
        for _ in range(cfg.grad_accum):
            try: b = next(it)
            except StopIteration: it = iter(dl_tr); b = next(it)
            w = b.pop("ocr_weight", None)
            b = {k: v.to(DEVICE, non_blocking=True) for k, v in b.items()}
            if w is not None: w = w.to(DEVICE)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
                out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"])
                loss = masked_mlm_loss(out.logits.float(), b["labels"], w) / cfg.grad_accum
            scaler.scale(loss).backward()
            running.append(loss.item() * cfg.grad_accum)
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sch.step()

        if (step + 1) % cfg.log_every == 0:
            m = float(np.mean(running[-cfg.log_every * cfg.grad_accum:]))
            c = sampler.competence(step * cfg.grad_accum)
            pbar.set_postfix(loss=f"{m:.3f}", ppl=f"{math.exp(min(m,20)):.1f}", c=f"{c:.2f}")
            history.append({"step": step + 1, "train_loss": m, "competence": c,
                            "phase": "embed" if step < warm_steps else "full",
                            "lr": sch.get_last_lr()[0]})
        if (step + 1) % cfg.eval_every == 0 or step + 1 == total_steps:
            el = eval_mlm(model, dl_ev, tok)
            history.append({"step": step + 1, "eval_loss": el, "eval_ppl": math.exp(min(el, 20))})
            log(f"[{arm}] step {step+1}  eval_loss={el:.4f}  ppl={math.exp(min(el,20)):.2f}")
            if el < best_loss - 1e-4:
                best_loss, since_improve = el, 0
                best_state = {k: v.detach().cpu().clone() for k, v in
                              (model.module if isinstance(model, nn.DataParallel)
                               else model).state_dict().items()}
            elif step >= warm_steps:      # never early-stop during phase 1
                since_improve += 1
                if since_improve >= cfg.early_stop_patience:
                    log(f"[{arm}] early stop at step {step+1} "
                        f"(no improvement in {since_improve} evals; best={best_loss:.4f})")
                    break
        if (step + 1) % cfg.save_every == 0 or step + 1 == total_steps:
            torch.save({"model": (model.module if isinstance(model, nn.DataParallel) else model).state_dict(),
                        "opt": opt.state_dict(), "sch": sch.state_dict(),
                        "scaler": scaler.state_dict(), "step": step + 1}, ck)
            (arm_dir / "history.json").write_text(jdump(history))
        if budget_exhausted():
            log(f"[{arm}] session budget reached at step {step+1} — checkpointed, re-run to resume")
            torch.save({"model": (model.module if isinstance(model, nn.DataParallel) else model).state_dict(),
                        "opt": opt.state_dict(), "sch": sch.state_dict(),
                        "scaler": scaler.state_dict(), "step": step + 1}, ck)
            (arm_dir / "history.json").write_text(jdump(history))
            raise TimeoutError(f"budget_exhausted at step {step+1} in arm {arm}")

    core = model.module if isinstance(model, nn.DataParallel) else model
    if best_state is not None:            # always export the best checkpoint,
        core.load_state_dict(best_state)  # not whatever the last step happened to be
        log(f"[{arm}] restored best checkpoint (eval_loss={best_loss:.4f})")
    core.save_pretrained(arm_dir, safe_serialization=True); tok.save_pretrained(arm_dir)
    (arm_dir / "history.json").write_text(jdump(history))
    history.append({"final_best_eval_loss": best_loss,
                    "steps_run": int(locals().get("step", start_step - 1)) + 1,
                    "total_steps_planned": total_steps})
    (arm_dir / "history.json").write_text(jdump(history))
    del model, opt, sch, scaler; gc.collect(); torch.cuda.empty_cache()
    return arm_dir

log("pretraining loop defined")


---
## Section 6b — Choosing the base encoder empirically

Bangla encoder work splits into three families, and which one to continue from is an empirical
question we should not answer by assumption:

| family | examples | objective | usable for MLM-DAPT? |
|---|---|---|---|
| Massively multilingual | XLM-R, mBERT | MLM | yes |
| Indic-focused multilingual | MuRIL, IndicBERT | MLM (+ transliteration) | yes |
| Bangla monolingual | **BanglaBERT** (Bangla2B+, 27.5 GB), BanglishBERT | **ELECTRA replaced-token detection** | **no MLM head** |

BanglaBERT is the strongest reported Bangla encoder (BLUB 77.09 vs XLM-R-large 76.79, mBERT 70.29),
and BanglishBERT is bilingual Bangla+English — a natural fit for this corpus. But both are ELECTRA
*discriminators*: they have no MLM head to continue training, so MLM-based DAPT cannot start from
them without switching the whole objective to RTD. We therefore restrict candidates to MLM models
and **note this as a limitation**, with an ELECTRA-objective variant as clearly-scoped future work.

Each candidate gets an identical short probe DAPT with its own native tokenizer; the winner on
held-out pure-mask MLM loss becomes the base for every curriculum arm. The selection table itself
belongs in the paper — "we chose X because it wins on held-out loss" is a far stronger sentence
than "we used XLM-R".


In [ ]:
# ============================================================================
# CELL 16b — Base-encoder selection (identical probe DAPT per candidate)
# ============================================================================
with stage("base_select"):
    if not stage_done("base_select"):
        rows = []
        for cand in cfg.base_candidates:
            try:
                init = build_init(cand, vocab_transfer=False)
                d = pretrain_arm("probe", init_dir=init,
                                 steps_override=cfg.base_select_steps,
                                 out_name=f"probe_{cand.split('/')[-1]}",
                                 mode_override="random")
                hist = json.loads((d / "history.json").read_text())
                ev = [h["eval_loss"] for h in hist if "eval_loss" in h]
                tk = AutoTokenizer.from_pretrained(init)
                bn = df.loc[df.language == "bn", "text"].head(400).tolist()
                rows.append({"base_model": cand,
                             "best_eval_loss": min(ev) if ev else float("nan"),
                             "eval_ppl": math.exp(min(min(ev), 20)) if ev else float("nan"),
                             "vocab_size": int(tk.vocab_size),
                             "fertility_bn": fertility_stats(tk, bn)["fertility"],
                             "cluster_split_bn": cluster_split_rate(tk, bn)})
                log(f"base probe {cand}: eval_loss={rows[-1]['best_eval_loss']:.4f}")
            except Exception as e:
                log(f"base probe FAILED for {cand}: {type(e).__name__}: {e}")
        save_art(pd.DataFrame(rows), "base_selection.csv")

base_sel = load_art("base_selection.csv")
if base_sel is not None and len(base_sel):
    base_sel = base_sel.sort_values("best_eval_loss")
    display(base_sel.round(4))
    cfg.base_model = base_sel.iloc[0]["base_model"]
    log(f"SELECTED base encoder: {cfg.base_model}")
else:
    log(f"base selection produced nothing — falling back to {cfg.base_model}")

# the starting checkpoints every curriculum arm will use
INIT_DIR  = build_init(cfg.base_model, vocab_transfer=False)   # native vocabulary
INIT_FOCUS = None
log(f"INIT_DIR = {INIT_DIR}")


In [ ]:
# ============================================================================
# CELL 17 — Run the arms.  Comment arms out to spread across Kaggle sessions.
# ============================================================================
# `vocab_swap` = grade_cl curriculum but starting from the FOCUS-transferred NCTB
# vocabulary. Isolating it as its own arm keeps the curriculum claim (C1) from
# being confounded with the vocabulary claim (C3).
ARMS = ["grade_cl", "random", "anti", "length_cl", "vocab_swap", "no_dapt"]
if cfg.smoke_test: ARMS = ["grade_cl", "random", "vocab_swap", "no_dapt"]

ARM_PATHS = {}
for arm in ARMS:
    st = f"pretrain_{arm}"
    if stage_done(st):
        ARM_PATHS[arm] = OUT / "models" / f"nctb_bert_{arm}"
        log(f"stage '{st}' already done — skipping"); continue
    if budget_exhausted(0.5):
        log(f"stopping before arm '{arm}': session budget exhausted. Re-run the notebook to continue.")
        break
    try:
        if arm == "vocab_swap":
            init = build_init(cfg.base_model, vocab_transfer=True)
            ARM_PATHS[arm] = pretrain_arm(arm, init_dir=init, mode_override="grade_cl")
        else:
            ARM_PATHS[arm] = pretrain_arm(arm, init_dir=INIT_DIR)
        mark_stage(st)
    except TimeoutError as e:
        log(str(e)); break

# collect whichever arms are finished
for arm in ARMS:
    p = OUT / "models" / f"nctb_bert_{arm}"
    if (p / "config.json").exists(): ARM_PATHS[arm] = p
log("arms available: " + ", ".join(ARM_PATHS))


In [ ]:
# ============================================================================
# CELL 18 — Figure 5: pretraining dynamics across curriculum arms
# ============================================================================
hists = {}
for arm, p in ARM_PATHS.items():
    hp = p / "history.json"
    if hp.exists(): hists[arm] = pd.DataFrame(json.loads(hp.read_text()))

if hists:
    LABEL = {"grade_cl": "GRADE-CL (ours)", "random": "random (standard DAPT)",
             "anti": "anti-curriculum", "length_cl": "length proxy CL",
             "vocab_swap": "GRADE-CL + NCTB vocab", "no_dapt": "no DAPT"}
    fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.4))
    ax = axes[0]
    for i, arm in enumerate([a for a in ARMS if a in hists and a != "no_dapt"][:4]):
        h = hists[arm].dropna(subset=["train_loss"])
        if h.empty: continue
        y = h["train_loss"].rolling(5, min_periods=1).mean()
        ax.plot(h["step"], y, color=PAL[i], label=LABEL.get(arm, arm))
        ax.text(h["step"].iloc[-1], y.iloc[-1], "  " + LABEL.get(arm, arm).split(" ")[0],
                color=PAL[i], fontsize=8, va="center")
    ax.legend(loc="upper right")
    finish(ax, "MLM training loss", "5-point rolling mean", "optimisation step", "loss")

    ax = axes[1]
    for i, arm in enumerate([a for a in ARMS if a in hists and a != "no_dapt"][:4]):
        h = hists[arm].dropna(subset=["eval_loss"])
        if h.empty: continue
        ax.plot(h["step"], h["eval_ppl"], color=PAL[i], marker="o", ms=4,
                label=LABEL.get(arm, arm))
    ax.legend(loc="upper right")
    finish(ax, "Held-out pseudo-perplexity", "unseen books (ds_val)", "optimisation step", "perplexity")
    fig.tight_layout(); savefig(fig, "fig5_pretraining_dynamics"); plt.show()
else:
    print("no training history yet — run Cell 17 first")


---
## Section 7 — Intrinsic evaluation

* **Pseudo-perplexity (PPPL)** on held-out books, computed by masking each token in turn in strided
  batches (Salazar et al., 2020). Reported per language and per class band — the band breakdown is
  what shows whether GRADE-CL actually helps the *hard* end of the curriculum or just the easy end.
* **OCR-robustness curve.** We inject synthetic OCR noise at rates $\epsilon\in\{0,0.05,\dots,0.25\}$
  using a confusion model estimated from the corpus's own dual-engine disagreements, and measure the
  relative PPPL degradation. A model trained on genuinely noisy educational text should degrade more
  slowly than one trained on clean web text.


In [ ]:
# ============================================================================
# CELL 19 — Pseudo-perplexity + OCR robustness
# ============================================================================
@torch.no_grad()
def pseudo_perplexity(model_path, texts, max_len=256, batch=16, stride_mask=8, limit=300):
    """Strided PPPL: mask every stride_mask-th position, average NLL over masked tokens."""
    tk = AutoTokenizer.from_pretrained(model_path)
    md_ = AutoModelForMaskedLM.from_pretrained(model_path).to(DEVICE).eval()
    if md_.config.vocab_size != len(tk): log(f"warn: vocab mismatch for {model_path}")
    nll, ntok = 0.0, 0
    texts = texts[:limit]
    for i in range(0, len(texts), batch):
        chunk = texts[i:i+batch]
        enc = tk(chunk, truncation=True, max_length=max_len, padding=True, return_tensors="pt")
        ids = enc["input_ids"].to(DEVICE); am = enc["attention_mask"].to(DEVICE)
        for off in range(stride_mask):
            pos = torch.zeros_like(ids, dtype=torch.bool)
            pos[:, off::stride_mask] = True
            pos &= am.bool()
            for sid in tk.all_special_ids: pos &= ids != sid
            if pos.sum() == 0: continue
            mi = ids.clone(); mi[pos] = tk.mask_token_id
            lab = torch.full_like(ids, -100); lab[pos] = ids[pos]
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
                logits = md_(input_ids=mi, attention_mask=am).logits.float()
            l = F.cross_entropy(logits.view(-1, logits.size(-1)), lab.view(-1),
                                ignore_index=-100, reduction="sum")
            nll += l.item(); ntok += int(pos.sum())
    del md_; gc.collect(); torch.cuda.empty_cache()
    return math.exp(nll / max(ntok, 1))

# ---- corpus-derived OCR confusion model -----------------------------------
def build_ocr_confusion(max_pages=400) -> Dict[str, List[str]]:
    """Estimate character confusions from Surya vs Tesseract disagreement in raw_ocr/."""
    import difflib
    conf = defaultdict(Counter)
    files = []
    for band in BANDS:
        rd = DATA_ROOT / band / "raw_ocr"
        if rd.exists(): files += sorted(rd.glob("*.json"))
    rng = np.random.default_rng(cfg.seed); rng.shuffle(files)
    seen = 0
    for fp in files[:20]:
        try: obj = json.loads(Path(fp).read_text(encoding="utf-8"))
        except Exception: continue
        for pg in obj.get("pages", []):
            a, b = (pg.get("surya") or "")[:1500], (pg.get("tesseract") or "")[:1500]
            if not a or not b: continue
            for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b, autojunk=False).get_opcodes():
                if tag == "replace" and (i2-i1) == (j2-j1) == 1:
                    conf[a[i1]][b[j1]] += 1
            seen += 1
            if seen >= max_pages: break
        if seen >= max_pages: break
    return {k: [c for c, _ in v.most_common(6)] for k, v in conf.items() if sum(v.values()) >= 3}

def inject_ocr_noise(t: str, eps: float, conf: Dict[str, List[str]], rng) -> str:
    if eps <= 0: return t
    out = []
    for ch in t:
        if rng.random() < eps:
            if ch in conf and conf[ch]: out.append(rng.choice(conf[ch])); continue
            r = rng.random()
            if r < 0.4: continue                       # deletion
            if r < 0.7: out.append(ch); out.append(ch) # duplication
            else: out.append(" ")                      # segmentation error
        else: out.append(ch)
    return "".join(out)

with stage("intrinsic"):
    if not stage_done("intrinsic"):
        OCR_CONF = build_ocr_confusion()
        save_art({k: v for k, v in list(OCR_CONF.items())[:500]}, "ocr_confusion.json")
        test_txt = df[df.split == "ds_test"]
        rows = []
        eval_models = {f"NCTB-BERT[{a}]": str(p) for a, p in ARM_PATHS.items()}
        for b in cfg.baselines:
            eval_models[b] = b
        rng = np.random.default_rng(cfg.seed)
        for name, path in eval_models.items():
            try:
                for lang in ["bn", "en"]:
                    tx = test_txt.loc[test_txt.language == lang, "text"].tolist()
                    if not tx: continue
                    tx = list(rng.choice(tx, size=min(200, len(tx)), replace=False))
                    base = pseudo_perplexity(path, tx)
                    rows.append({"model": name, "language": lang, "eps": 0.0, "pppl": base})
                    for eps in ([0.10, 0.20] if not cfg.smoke_test else [0.10]):
                        noisy = [inject_ocr_noise(t, eps, OCR_CONF, rng) for t in tx]
                        rows.append({"model": name, "language": lang, "eps": eps,
                                     "pppl": pseudo_perplexity(path, noisy)})
                log(f"intrinsic done: {name}")
            except Exception as e:
                log(f"intrinsic FAILED for {name}: {e}")
        save_art(pd.DataFrame(rows), "intrinsic.csv")

intrinsic = load_art("intrinsic.csv")
display(intrinsic[intrinsic.eps == 0].pivot(index="model", columns="language", values="pppl").round(2))


---
## Section 8 — NCTBench-Eval: four downstream tasks (contribution C4)

All four are built from dataset metadata, cost zero annotation, and are split **by book**.

| Task | Type | Labels from | Metric |
|---|---|---|---|
| **T1 Grade-band** | 5-way sentence classification | `class_band` | macro-F1 |
| **T2 Subject** | $k$-way sentence classification | `subject` (top-12 subjects) | macro-F1 |
| **T3 Chapter-boundary** | binary sentence-pair | `next_chunk_id` adjacency | MCC |
| **T4 Passage retrieval** | frozen-encoder dense retrieval | chapter membership | MRR\@10, R\@10 |

T1 is the task the curriculum contribution most directly predicts an effect on. T3 is included
because it needs *discourse-level* representation, not topical keywords. T4 uses **frozen** encoders,
so it isolates representation quality from fine-tuning capacity.


In [ ]:
# ============================================================================
# CELL 20 — Build the four task datasets
# ============================================================================
def build_tasks(frame: pd.DataFrame, split_col="split") -> Dict[str, Dict[str, pd.DataFrame]]:
    tasks = {}
    d = frame[frame[split_col].str.startswith("ds_")].copy()

    # ---- T1 grade band ----
    t1 = d[["text", "class_band", split_col, "book_id"]].rename(columns={"class_band": "label_raw"})
    # A class present in test but absent from train is untrainable by construction
    # and silently caps macro-F1 for every model. Drop it and say so.
    trainable = set(t1.loc[t1[split_col] == "ds_train", "label_raw"])
    missing = sorted(set(t1["label_raw"]) - trainable)
    if missing:
        log(f"T1: dropping class band(s) {missing} — no training books available for them")
        t1 = t1[t1.label_raw.isin(trainable)]
    tasks["T1_grade_band"] = {"data": t1, "kind": "single"}

    # ---- T2 subject (top-12) ----
    top = d["subject"].value_counts().head(12).index
    t2 = d[d.subject.isin(top)][["text", "subject", split_col, "book_id"]].rename(
        columns={"subject": "label_raw"})
    tasks["T2_subject"] = {"data": t2, "kind": "single"}

    # ---- T3 chapter boundary (sentence pair) ----
    by_id = d.set_index("chunk_id")["text"].to_dict()
    rng = np.random.default_rng(cfg.seed)
    rows = []
    for sp, g in d.groupby(split_col):
        pool = g["chunk_id"].tolist()
        for _, r in g.iterrows():
            nxt = r.get("next_chunk_id")
            if isinstance(nxt, str) and nxt in by_id:
                rows.append({"text": r["text"], "text_b": by_id[nxt], "label_raw": "adjacent",
                             split_col: sp, "book_id": r["book_id"]})
                neg = by_id[pool[int(rng.integers(len(pool)))]]
                if neg != by_id[nxt]:
                    rows.append({"text": r["text"], "text_b": neg, "label_raw": "not_adjacent",
                                 split_col: sp, "book_id": r["book_id"]})
    tasks["T3_chapter_boundary"] = {"data": pd.DataFrame(rows), "kind": "pair"}

    # ---- T4 retrieval (frozen, test split only) ----
    t4 = d[d[split_col] == "ds_test"][["chunk_id", "text", "book_id", "chapter_no", "subject"]].copy()
    t4["group"] = t4["book_id"] + "|ch" + t4["chapter_no"].astype(str)
    grp = t4["group"].value_counts()
    t4 = t4[t4["group"].isin(grp[grp >= 4].index)]
    tasks["T4_retrieval"] = {"data": t4, "kind": "retrieval"}
    return tasks

with stage("tasks"):
    if not stage_done("tasks"):
        TASKS = build_tasks(df)
        for k, v in TASKS.items(): save_art(v["data"], f"task_{k}.parquet")
        save_art({k: v["kind"] for k, v in TASKS.items()}, "task_kinds.json")

TASK_KINDS = load_art("task_kinds.json")
TASKS = {k: {"data": load_art(f"task_{k}.parquet"), "kind": v} for k, v in TASK_KINDS.items()}
for k, v in TASKS.items():
    d = v["data"]
    if v["kind"] == "retrieval":
        print(f"{k:24s} {len(d):6,} passages · {d.group.nunique()} chapters")
    else:
        print(f"{k:24s} {len(d):6,} ex · {d.label_raw.nunique()} classes · "
              f"splits={dict(d['split'].value_counts())}")


In [ ]:
# ============================================================================
# CELL 21 — Fine-tuning + evaluation harness (multi-seed)
# ============================================================================
class ClfDataset(TorchDataset):
    def __init__(self, frame, tok, max_len, label2id, pair=False):
        self.a = frame["text"].tolist()
        self.b = frame["text_b"].tolist() if pair else None
        self.y = [label2id[l] for l in frame["label_raw"]]
        self.tok = tok; self.L = max_len
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        if self.b is None:
            e = self.tok(self.a[i], truncation=True, max_length=self.L)
        else:
            e = self.tok(self.a[i], self.b[i], truncation=True, max_length=self.L)
        e["labels"] = self.y[i]
        return e

def collate_clf(feats, tok):
    labels = torch.tensor([f.pop("labels") for f in feats])
    b = tok.pad(feats, return_tensors="pt", pad_to_multiple_of=8)
    b["labels"] = labels
    return b

def finetune_eval(model_path, task_name, task, seed, split_col="split") -> Dict[str, Any]:
    d = task["data"]; pair = task["kind"] == "pair"
    labels = sorted(d["label_raw"].unique()); l2i = {l: i for i, l in enumerate(labels)}
    set_all_seeds(seed)
    kw = {"cache_dir": cfg.model_cache} if cfg.model_cache else {}
    tok = AutoTokenizer.from_pretrained(model_path, **kw)
    mdl = AutoModelForSequenceClassification.from_pretrained(
        model_path, num_labels=len(labels), **kw).to(DEVICE)
    if N_GPU > 1: mdl = nn.DataParallel(mdl)

    parts = {s: d[d[split_col] == s] for s in ["ds_train", "ds_val", "ds_test"]}
    empty = [s for s, v in parts.items() if len(v) == 0]
    if empty:
        raise ValueError(
            f"{task_name}: split(s) {empty} are empty, so there is nothing to fine-tune on. "
            f"This comes from the book-level split in Cell 8, not from this task. "
            f"Check CFG.downstream_book_frac / ds_val_frac / ds_test_frac.")
    if cfg.smoke_test:
        parts = {k: v.sample(min(len(v), 600), random_state=seed) for k, v in parts.items()}
    dls = {}
    for s, fr in parts.items():
        ds = ClfDataset(fr, tok, cfg.ft_max_len, l2i, pair)
        dls[s] = DataLoader(ds, batch_size=cfg.ft_bs * max(N_GPU, 1),
                            shuffle=(s == "ds_train"), num_workers=2,
                            collate_fn=lambda f: collate_clf(f, tok), drop_last=False)

    steps = max(1, len(dls["ds_train"]) * cfg.ft_epochs)
    opt = torch.optim.AdamW(mdl.parameters(), lr=cfg.ft_lr, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = make_scaler(torch.cuda.is_available())

    def run_eval(loader):
        mdl.eval(); P, Y = [], []
        with torch.no_grad():
            for b in loader:
                y = b.pop("labels")
                b = {k: v.to(DEVICE) for k, v in b.items()}
                with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
                    lg = mdl(**b).logits.float()
                P.append(lg.argmax(-1).cpu()); Y.append(y)
        mdl.train()
        return torch.cat(P).numpy(), torch.cat(Y).numpy()

    best_val, best_state = -1, None
    for ep in range(cfg.ft_epochs):
        for b in dls["ds_train"]:
            opt.zero_grad(set_to_none=True)
            b = {k: v.to(DEVICE) for k, v in b.items()}
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
                loss = mdl(**b).loss
                if loss.dim() > 0: loss = loss.mean()
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
        p, y = run_eval(dls["ds_val"])
        v = f1_score(y, p, average="macro")
        if v > best_val:
            best_val = v
            best_state = {k: t.detach().cpu().clone() for k, t in
                          (mdl.module if isinstance(mdl, nn.DataParallel) else mdl).state_dict().items()}
    if best_state is not None:
        (mdl.module if isinstance(mdl, nn.DataParallel) else mdl).load_state_dict(best_state)
    p, y = run_eval(dls["ds_test"])
    res = {"model": str(model_path), "task": task_name, "seed": seed,
           "macro_f1": f1_score(y, p, average="macro"),
           "accuracy": accuracy_score(y, p),
           "mcc": matthews_corrcoef(y, p),
           "val_macro_f1": best_val,
           "n_test": int(len(y))}
    res["_preds"] = p.tolist(); res["_gold"] = y.tolist()
    del mdl, opt, sch, scaler, best_state; gc.collect(); torch.cuda.empty_cache()
    return res

@torch.no_grad()
def retrieval_eval(model_path, task, k=10) -> Dict[str, Any]:
    d = task["data"]
    kw = {"cache_dir": cfg.model_cache} if cfg.model_cache else {}
    tok = AutoTokenizer.from_pretrained(model_path, **kw)
    enc = AutoModel.from_pretrained(model_path, **kw).to(DEVICE).eval()
    texts = d["text"].tolist(); groups = d["group"].values
    embs = []
    for i in range(0, len(texts), 64):
        b = tok(texts[i:i+64], truncation=True, max_length=cfg.ft_max_len,
                padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
            h = enc(**b).last_hidden_state.float()
        m = b["attention_mask"].unsqueeze(-1).float()
        embs.append(((h * m).sum(1) / m.sum(1)).cpu())
    E = F.normalize(torch.cat(embs), dim=-1)
    S = E @ E.T; S.fill_diagonal_(-1e4)
    topk = S.topk(k, dim=-1).indices.numpy()
    rel = (groups[topk] == groups[:, None])
    rr = np.where(rel.any(1), 1.0 / (rel.argmax(1) + 1), 0.0)
    gsize = pd.Series(groups).map(pd.Series(groups).value_counts()).values - 1
    recall = rel.sum(1) / np.minimum(np.maximum(gsize, 1), k)
    del enc; gc.collect(); torch.cuda.empty_cache()
    return {"model": str(model_path), "task": "T4_retrieval", "seed": 0,
            "mrr@10": float(rr.mean()), "recall@10": float(recall.mean()),
            "n_test": int(len(texts))}

log("evaluation harness ready")


In [ ]:
# ============================================================================
# CELL 22 — Run every model × task × seed  (resumable, budget-aware)
# ============================================================================
MODEL_ZOO = {f"NCTB-BERT[{a}]": str(p) for a, p in ARM_PATHS.items()}
for b in cfg.baselines: MODEL_ZOO[b] = b

RES_PATH = OUT / "artifacts" / "downstream_results.jsonl"
done = set()
if RES_PATH.exists():
    for line in RES_PATH.read_text().splitlines():
        r = json.loads(line); done.add((r["model"], r["task"], r["seed"]))

clf_tasks = [t for t, v in TASKS.items() if v["kind"] in ("single", "pair")]
with open(RES_PATH, "a") as fout:
    for mname, mpath in MODEL_ZOO.items():
        for tname in clf_tasks:
            for seed in cfg.seeds:
                if (mname, tname, seed) in done: continue
                if budget_exhausted(0.3):
                    log("budget exhausted — re-run the notebook to continue downstream runs"); break
                try:
                    r = finetune_eval(mpath, tname, TASKS[tname], seed)
                    r["model"] = mname
                    fout.write(jdump(r) + "\n"); fout.flush()
                    done.add((mname, tname, seed))
                    log(f"{mname:28s} {tname:22s} seed={seed:<6d} macroF1={r['macro_f1']:.4f}")
                except Exception as e:
                    log(f"FAILED {mname}/{tname}/{seed}: {type(e).__name__}: {e}")
        if (mname, "T4_retrieval", 0) not in done and not budget_exhausted(0.2):
            try:
                r = retrieval_eval(mpath, TASKS["T4_retrieval"]); r["model"] = mname
                fout.write(jdump(r) + "\n"); fout.flush()
                done.add((mname, "T4_retrieval", 0))
                log(f"{mname:28s} T4_retrieval           MRR@10={r['mrr@10']:.4f}")
            except Exception as e:
                log(f"FAILED {mname}/T4: {e}")

results = pd.DataFrame([json.loads(l) for l in RES_PATH.read_text().splitlines()])
save_art(results.drop(columns=[c for c in ["_preds", "_gold"] if c in results], errors="ignore"),
         "downstream_results.csv")
if "macro_f1" in results.columns and results["macro_f1"].notna().any():
    display(results.groupby(["task", "model"])["macro_f1"].mean().unstack(0).round(4))
else:
    log("NO classification results were produced — every fine-tuning run failed. "
        "Scroll up for the first 'FAILED ...' line; that message names the real cause.")
if "mrr@10" in results.columns and results["mrr@10"].notna().any():
    display(results.dropna(subset=["mrr@10"])[["model", "mrr@10", "recall@10"]].round(4))


---
## Section 9 — Statistical protocol

For a Q1 venue, a single number per cell is not enough. We report, for every model × task:

* **mean ± s.d.** over 5 random seeds;
* a **BCa-free percentile bootstrap 95 % CI** over the test set (10,000 resamples), pooling seeds;
* a **paired bootstrap significance test** of NCTB-BERT(GRADE-CL) against the strongest baseline and
  against each ablation arm, resampling test instances with the seed-matched prediction pairs held together;
* **Holm–Bonferroni** correction across the family of comparisons within each task.

We additionally report the **leakage inflation** $\Delta = F_1^{\text{chunk-split}} - F_1^{\text{book-split}}$
to quantify contribution C4.


In [ ]:
# ============================================================================
# CELL 23 — Bootstrap CIs, paired bootstrap tests, Holm correction
# ============================================================================
def bootstrap_ci(y, p, metric="macro_f1", n=None, alpha=0.05, seed=0):
    n = n or cfg.bootstrap_n
    rng = np.random.default_rng(seed); y = np.asarray(y); p = np.asarray(p)
    fn = (lambda a, b: f1_score(a, b, average="macro")) if metric == "macro_f1" else \
         (lambda a, b: matthews_corrcoef(a, b))
    stats_ = np.empty(n)
    for i in range(n):
        idx = rng.integers(0, len(y), len(y))
        stats_[i] = fn(y[idx], p[idx])
    lo, hi = np.percentile(stats_, [100*alpha/2, 100*(1-alpha/2)])
    return float(fn(y, p)), float(lo), float(hi)

def paired_bootstrap(y, pA, pB, metric="macro_f1", n=None, seed=0):
    """H0: model A is not better than B. Returns (delta, one-sided p)."""
    n = n or cfg.bootstrap_n
    rng = np.random.default_rng(seed)
    y = np.asarray(y); pA = np.asarray(pA); pB = np.asarray(pB)
    fn = (lambda a, b: f1_score(a, b, average="macro")) if metric == "macro_f1" else \
         (lambda a, b: matthews_corrcoef(a, b))
    obs = fn(y, pA) - fn(y, pB)
    cnt = 0
    for i in range(n):
        idx = rng.integers(0, len(y), len(y))
        d = fn(y[idx], pA[idx]) - fn(y[idx], pB[idx])
        if d <= 0: cnt += 1
    return float(obs), float((cnt + 1) / (n + 1))

def holm(pvals: Dict[str, float], alpha=0.05) -> pd.DataFrame:
    items = sorted(pvals.items(), key=lambda kv: kv[1]); m = len(items)
    out, prev = [], 0.0
    for i, (k, p) in enumerate(items):
        thr = alpha / (m - i)
        adj = max(prev, min(1.0, p * (m - i))); prev = adj
        out.append({"comparison": k, "p_raw": p, "p_holm": adj, "significant": adj < alpha,
                    "threshold": thr})
    return pd.DataFrame(out)

with stage("stats"):
    if not stage_done("stats"):
        raw = [json.loads(l) for l in RES_PATH.read_text().splitlines()]
        clf = [r for r in raw if "_preds" in r]
        # aggregate: mean±sd across seeds
        agg = (pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in clf])
                 .groupby(["task", "model"])
                 .agg(macro_f1_mean=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
                      mcc_mean=("mcc", "mean"), mcc_sd=("mcc", "std"),
                      acc_mean=("accuracy", "mean"), n_seeds=("seed", "nunique"))
                 .reset_index())
        # pooled bootstrap CI on the first seed's predictions (representative)
        cis = []
        for (t, m), g in pd.DataFrame(clf).groupby(["task", "model"]):
            r0 = g.iloc[0]
            pt, lo, hi = bootstrap_ci(r0["_gold"], r0["_preds"], seed=cfg.seed)
            cis.append({"task": t, "model": m, "f1_point": pt, "ci_lo": lo, "ci_hi": hi})
        agg = agg.merge(pd.DataFrame(cis), on=["task", "model"], how="left")

        # paired tests: ours vs everything else, per task
        OURS = "NCTB-BERT[grade_cl]"
        tests = []
        for t, g in pd.DataFrame(clf).groupby("task"):
            if OURS not in set(g.model): continue
            a = g[g.model == OURS].iloc[0]
            pv, block = {}, []
            for m in sorted(set(g.model) - {OURS}):
                b = g[g.model == m].iloc[0]
                if list(b["_gold"]) != list(a["_gold"]):
                    log(f"skip paired test {t}/{m}: gold order differs"); continue
                delta, p = paired_bootstrap(a["_gold"], a["_preds"], b["_preds"], seed=cfg.seed)
                pv[m] = p
                block.append({"task": t, "vs": m, "delta_f1": delta, "p_raw": p})
            if pv:
                h = holm(pv, cfg.alpha).set_index("comparison")
                for r in block:
                    r["p_holm"] = float(h.loc[r["vs"], "p_holm"])
                    r["significant"] = bool(h.loc[r["vs"], "significant"])
            tests += block
        save_art(agg, "results_aggregate.csv")
        save_art(pd.DataFrame(tests), "significance_tests.csv")

agg   = load_art("results_aggregate.csv")
tests = load_art("significance_tests.csv")
display(agg.pivot(index="model", columns="task", values="macro_f1_mean").round(4))
display(tests.round(4) if tests is not None and len(tests) else "no paired tests yet")


In [ ]:
# ============================================================================
# CELL 24 — Leakage-inflation experiment (contribution C4)
# ============================================================================
with stage("leakage"):
    if not stage_done("leakage") and "grade_cl" in ARM_PATHS:
        TASKS_CHUNK = build_tasks(df, split_col="split_chunklevel")
        rows = []
        for tname in ["T1_grade_band", "T2_subject"]:
            for seed in cfg.seeds[:2]:
                r = finetune_eval(str(ARM_PATHS["grade_cl"]), tname, TASKS_CHUNK[tname],
                                  seed, split_col="split_chunklevel")
                rows.append({"task": tname, "seed": seed, "split": "chunk-level",
                             "macro_f1": r["macro_f1"]})
        book = (pd.DataFrame([json.loads(l) for l in RES_PATH.read_text().splitlines()])
                  .query("model == 'NCTB-BERT[grade_cl]'")
                  .loc[lambda x: x.task.isin(["T1_grade_band", "T2_subject"]),
                       ["task", "seed", "macro_f1"]].assign(split="book-level"))
        lk = pd.concat([pd.DataFrame(rows), book], ignore_index=True)
        save_art(lk, "leakage_experiment.csv")

leak = load_art("leakage_experiment.csv")
if leak is not None:
    piv = leak.groupby(["task", "split"])["macro_f1"].mean().unstack()
    piv["inflation"] = piv.get("chunk-level", 0) - piv.get("book-level", 0)
    display(piv.round(4))


In [ ]:
# ============================================================================
# CELL 25 — Figures 6-8: main results, ablations, per-band gains
# ============================================================================
NICE = {"NCTB-BERT[grade_cl]": "NCTB-BERT (GRADE-CL)", "NCTB-BERT[random]": "NCTB-BERT (random DAPT)",
        "NCTB-BERT[anti]": "NCTB-BERT (anti-CL)", "NCTB-BERT[length_cl]": "NCTB-BERT (length CL)",
        "NCTB-BERT[vocab_swap]": "NCTB-BERT (+NCTB vocab)", "NCTB-BERT[no_dapt]": "NCTB-BERT (no DAPT)"}
def nice(m): return NICE.get(m, m.split("/")[-1])

# ---- Figure 6: main results, ours vs baselines, with 95% CI ---------------
main_models = ["NCTB-BERT[grade_cl]"] + [b for b in cfg.baselines]
sub = agg[agg.model.isin(main_models)]
tasks_sorted = sorted(sub.task.unique())
if len(sub):
    fig, axes = plt.subplots(1, len(tasks_sorted), figsize=(3.4*len(tasks_sorted), 3.5), squeeze=False)
    for j, t in enumerate(tasks_sorted):
        ax = axes[0][j]
        g = sub[sub.task == t].sort_values("macro_f1_mean")
        y = np.arange(len(g))
        cols = [PAL[0] if m.startswith("NCTB") else MUTED for m in g.model]
        ax.barh(y, g.macro_f1_mean, 0.6, color=cols)
        err_lo = (g.macro_f1_mean - g.ci_lo.fillna(g.macro_f1_mean)).clip(lower=0)
        err_hi = (g.ci_hi.fillna(g.macro_f1_mean) - g.macro_f1_mean).clip(lower=0)
        ax.errorbar(g.macro_f1_mean, y, xerr=[err_lo, err_hi], fmt="none",
                    ecolor=INK2, elinewidth=1.0, capsize=2.5)
        for yi, v in zip(y, g.macro_f1_mean): ax.text(v + 0.012, yi, f"{v:.3f}", va="center",
                                                      fontsize=7.3, color=INK2)
        ax.set_yticks(y); ax.set_yticklabels([nice(m) for m in g.model], fontsize=7.8)
        ax.set_xlim(0, min(1.0, float(g.macro_f1_mean.max()) + 0.15))
        finish(ax, t.replace("_", " "), "macro-F1, 95% bootstrap CI", "macro-F1", None,
               ygrid=False, xgrid=True)
    fig.tight_layout(); savefig(fig, "fig6_main_results"); plt.show()

# ---- Figure 7: curriculum ablation ---------------------------------------
abl = agg[agg.model.str.startswith("NCTB-BERT")]
if len(abl):
    fig, ax = plt.subplots(figsize=(7.2, 3.4))
    arms_present = [f"NCTB-BERT[{a}]" for a in ARMS if f"NCTB-BERT[{a}]" in set(abl.model)]
    ts = sorted(abl.task.unique())
    x = np.arange(len(ts)); w = 0.8 / max(len(arms_present), 1)
    for i, m in enumerate(arms_present[:4]):
        g = abl[abl.model == m].set_index("task").reindex(ts)
        ax.bar(x + i*w - 0.4 + w/2, g.macro_f1_mean.values, w*0.92,
               yerr=g.macro_f1_sd.values, capsize=2,
               color=PAL[i], ecolor=INK2, error_kw={"elinewidth": 0.9},
               label=nice(m))
    ax.set_xticks(x); ax.set_xticklabels([t.replace("_", "\n") for t in ts], fontsize=8)
    ax.legend(ncols=2, loc="upper right")
    finish(ax, "Curriculum ablation", "macro-F1, mean ± s.d. over seeds", None, "macro-F1")
    fig.tight_layout(); savefig(fig, "fig7_ablation"); plt.show()

# ---- Figure 8: OCR robustness --------------------------------------------
if intrinsic is not None and intrinsic.eps.nunique() > 1:
    fig, ax = plt.subplots(figsize=(6.0, 3.4))
    base = intrinsic[intrinsic.eps == 0].groupby("model")["pppl"].mean()
    keep = ["NCTB-BERT[grade_cl]"] + list(cfg.baselines[:3])
    for i, m in enumerate([m for m in keep if m in set(intrinsic.model)][:4]):
        g = intrinsic[intrinsic.model == m].groupby("eps")["pppl"].mean()
        rel = g / base[m]
        ax.plot(rel.index, rel.values, color=PAL[i], marker="o", label=nice(m))
        ax.text(rel.index[-1], rel.values[-1], "  " + nice(m), color=PAL[i], fontsize=8, va="center")
    finish(ax, "Robustness to injected OCR noise",
           "pseudo-perplexity relative to clean text (lower = more robust)",
           "injected noise rate ε", "PPPL(ε) / PPPL(0)")
    fig.tight_layout(); savefig(fig, "fig8_ocr_robustness"); plt.show()


In [ ]:
# ============================================================================
# CELL 26 — LaTeX tables, model card, and export bundle
# ============================================================================
def to_latex(dfr, caption, label, float_fmt="%.3f"):
    return dfr.to_latex(index=False, escape=True, float_format=float_fmt,
                        caption=caption, label=label, position="t")

TAB = OUT / "tables"
if agg is not None and len(agg):
    t_main = (agg.assign(score=lambda d: d.macro_f1_mean.map("{:.3f}".format) + " ± " +
                         d.macro_f1_sd.fillna(0).map("{:.3f}".format))
                 .pivot(index="model", columns="task", values="score").reset_index())
    (TAB / "table2_main_results.tex").write_text(
        to_latex(t_main, "NCTBench-Eval results (macro-F1, mean ± s.d. over 5 seeds; "
                         "book-level splits).", "tab:main"))
if fert is not None:
    (TAB / "table1_tokenizer.tex").write_text(
        to_latex(fert.pivot(index="model", columns="language", values="fertility").reset_index(),
                 "Tokenizer fertility (subword tokens per word). Lower is better.", "tab:fert"))
if tests is not None and len(tests):
    (TAB / "table3_significance.tex").write_text(
        to_latex(tests, "Paired bootstrap tests of NCTB-BERT (GRADE-CL) against every other system, "
                        "Holm-corrected within task.", "tab:sig", "%.4f"))
if leak is not None:
    (TAB / "table4_leakage.tex").write_text(
        to_latex(leak.groupby(["task","split"])["macro_f1"].mean().reset_index(),
                 "Chunk-level vs book-level splitting: the inflation from near-duplicate leakage.",
                 "tab:leak"))

MODEL_CARD = f"""# NCTB-BERT

**Base:** {cfg.base_model} — selected empirically over {len(cfg.base_candidates)} MLM candidates by
held-out pure-mask MLM loss (see `base_selection.csv`), using its native vocabulary.
**Vocabulary variant:** a {cfg.vocab_size:,}-piece grapheme-cluster-constrained Unigram vocabulary
trained on NCTB with FOCUS embedding transfer is evaluated as the separate `vocab_swap` arm.
**Adaptation:** GRADE-CL — grade-referenced competence-based curriculum DAPT,
<= {cfg.max_steps_cap:,} steps ({cfg.target_epochs} target epochs, early-stopped), two-phase vocabulary transfer ({cfg.embed_warmup_frac:.0%} embeddings-only warmup), seq len {cfg.max_len}, MLM p={cfg.mlm_prob}, whole-word masking

## Corpus
NCTBench-v2 — {clean_report['n_after']:,} chunks from {df.book_id.nunique()} NCTB textbooks (classes 1-12,
Bengali + English). Pretraining pool = {int((df.split=='pretrain').sum()):,} chunks from books
disjoint from all evaluation books.

## Intended use
Bengali/English educational text understanding: readability and grade-level estimation, subject
routing, textbook passage retrieval, chapter segmentation, curriculum-aligned tutoring systems.

## Limitations
* Source text is OCR output; residual recognition errors persist by design (the model is trained to be
  robust to them, not to be trained on clean text).
* Textbook register only — expect domain shift on conversational or social-media Bengali.
* The grade signal reflects the Bangladeshi national curriculum specifically and may not transfer to
  other education systems.
* Chapter titles are frequently empty in the source metadata; chapter-level tasks rely on numbering.

## License
Corpus CC-BY-4.0. Model weights inherit the {cfg.base_model} license.
"""
(OUT / "MODEL_CARD.md").write_text(MODEL_CARD)

manifest = {
    "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
    "config": {k: str(v) for k, v in asdict(cfg).items()},
    "corpus": clean_report,
    "focus": focus_stats,
    "arms_trained": list(ARM_PATHS),
    "figures": sorted(p.name for p in (OUT/"figures").glob("*.pdf")),
    "tables": sorted(p.name for p in TAB.glob("*.tex")),
    "environment": {"torch": torch.__version__, "transformers": transformers.__version__,
                    "n_gpu": N_GPU},
}
(OUT / "RUN_MANIFEST.json").write_text(jdump(manifest, indent=2))

# bundle only the lightweight paper assets (never the model weights, and never
# into OUT itself -- that would recurse into the archive being written)
BUNDLE = Path("/kaggle/working/paper_bundle")
if BUNDLE.exists(): shutil.rmtree(BUNDLE)
BUNDLE.mkdir(parents=True)
for sub in ["figures", "tables", "artifacts", "logs"]:
    if (OUT / sub).exists(): shutil.copytree(OUT / sub, BUNDLE / sub)
for f in ["MODEL_CARD.md", "RUN_MANIFEST.json"]:
    if (OUT / f).exists(): shutil.copy(OUT / f, BUNDLE / f)
shutil.make_archive("/kaggle/working/nctb_bert_paper_bundle", "zip", root_dir=str(BUNDLE))
log("exported: figures/, tables/, MODEL_CARD.md, RUN_MANIFEST.json, "
    "/kaggle/working/nctb_bert_paper_bundle.zip")
print(MODEL_CARD)


---
## Section 10 — Writing the paper from these outputs

**Where each artifact goes:**

| Paper section | Artifact |
|---|---|
| §3.1 Corpus | `fig1_corpus_composition`, `corpus_summary.csv`, `cleaning_report.json` |
| §3.2 Difficulty index | `fig2_difficulty_index`, `proxy_validation.csv`, `difficulty_validation.json` |
| §3.3 Tokenizer | `fig3_tokenizer_fertility`, `table1_tokenizer.tex`, `focus_stats.json` |
| §3.4 GRADE-CL | `fig4_competence_schedule`, `fig5_pretraining_dynamics` |
| §4 Benchmark | `task_*.parquet` counts, split table |
| §5 Results | `fig6_main_results`, `table2_main_results.tex`, `table3_significance.tex` |
| §5.x Ablation | `fig7_ablation` |
| §5.y Robustness | `fig8_ocr_robustness`, `intrinsic.csv` |
| §5.z Leakage | `table4_leakage.tex` |

**Claims the pipeline is designed to support (state only what the numbers show):**

1. Expert grade labels are a *better* difficulty signal than the intrinsic proxies used in prior CL work —
   quantified by Figure 2b. If a proxy's $|\rho|$ with grade is low, that is a finding about the
   literature, not a defect of the corpus.
2. GRADE-CL > random DAPT on grade-sensitive tasks, and the anti-curriculum arm should be *worse* than
   random. If `anti` ≈ `grade_cl`, the effect is from re-ordering variance, not from difficulty — report
   that honestly; it is still a publishable negative result and reviewers will look for exactly this control.
3. Vocabulary specialisation cuts embedding parameters ~8× while lowering fertility (Table 1) — an
   efficiency claim independent of the curriculum claim.
4. Book-level splitting is necessary; chunk-level splitting inflates macro-F1 by $\Delta$ (Table 4).

**Threats to validity to pre-empt in the paper** (reviewers will raise these):

* *Grade ≠ difficulty.* Grade encodes curricular sequencing, which conflates conceptual difficulty with
  policy and subject availability. Mitigate by reporting the composite index's intrinsic components
  separately and showing the effect survives when $w_g$ is reduced (run a $w_g$ sweep as a robustness check).
* *Corpus size.* ~46k chunks is small for pretraining; that is precisely why DAPT-from-XLM-R rather than
  from-scratch is the right design. Say so explicitly.
* *OCR noise as a confound.* Report `mean_agreement` per band and check it does not correlate with the
  curriculum ordering, otherwise the curriculum is partly a noise curriculum. (`intrinsic.csv` + Figure 1b.)
* *Single pretraining seed per arm.* GPU budget permitting, run `grade_cl` and `random` at 2–3 pretraining
  seeds; otherwise state the limitation.

**Suggested extra ablation arms** (flip one flag each in `CFG`, then re-run — each adds a row to Figure 7):

* `use_ocr_weighted_mlm = True` — MLM loss weighted by dual-engine OCR agreement.
* `use_dual_view_contrastive = True` — Surya/Tesseract page pairs as naturally occurring positives.
* `difficulty_weights["grade"] = 0.0` — curriculum from intrinsic proxies only; isolates how much of the
  gain comes from the *expert* signal specifically. **This is the single most convincing ablation in the
  paper** — run it.
